In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # One-Notebook Setup — E-commerce Dynamic Hyper-Marketing Dashboard
# MAGIC
# MAGIC This notebook sets up everything in Databricks UI in one run:
# MAGIC
# MAGIC 1. Creates Unity Catalog schemas and volumes
# MAGIC 2. Ingests raw CSV files into Bronze Delta tables
# MAGIC 3. Builds typed Silver Delta tables
# MAGIC 4. Builds Gold dashboard tables
# MAGIC 5. Builds product push-now ranking and marketing action center
# MAGIC 6. Creates audit and data-quality tables
# MAGIC 7. Optimizes Delta tables
# MAGIC
# MAGIC ## Before running
# MAGIC Upload the CSV files to the raw volume path shown below, or set `UPLOAD_SAMPLE_DATA = True` to generate demo data directly from this notebook.

# COMMAND ----------

# ============================================================
# 0. Widgets / Parameters
# ============================================================

try:
    dbutils.widgets.text("catalog", "workspace")
    dbutils.widgets.text("schema_prefix", "ecommerce_hypermarketing_dev")
    dbutils.widgets.dropdown("mode", "batch", ["batch", "generate_sample_data"])
    dbutils.widgets.text("stock_low_hrs", "12")
    dbutils.widgets.text("min_push_roas", "5.0")
    dbutils.widgets.text("min_push_cvr", "4.5")
except Exception:
    pass

In [0]:


CATALOG = dbutils.widgets.get("catalog")
SCHEMA_PREFIX = dbutils.widgets.get("schema_prefix")
MODE = dbutils.widgets.get("mode")
STOCK_LOW_HRS = float(dbutils.widgets.get("stock_low_hrs"))
MIN_PUSH_ROAS = float(dbutils.widgets.get("min_push_roas"))
MIN_PUSH_CVR = float(dbutils.widgets.get("min_push_cvr"))

BRONZE = f"{CATALOG}.{SCHEMA_PREFIX}_bronze"
SILVER = f"{CATALOG}.{SCHEMA_PREFIX}_silver"
GOLD = f"{CATALOG}.{SCHEMA_PREFIX}_gold"
OPS = f"{CATALOG}.{SCHEMA_PREFIX}_ops"

RAW_VOLUME = f"/Volumes/{CATALOG}/{SCHEMA_PREFIX}_bronze/ecommerce_raw"
CHECKPOINT_VOLUME = f"/Volumes/{CATALOG}/{SCHEMA_PREFIX}_bronze/ecommerce_checkpoint"
SCHEMA_VOLUME = f"/Volumes/{CATALOG}/{SCHEMA_PREFIX}_bronze/ecommerce_schema"

print("Catalog:", CATALOG)
print("Bronze schema:", BRONZE)
print("Silver schema:", SILVER)
print("Gold schema:", GOLD)
print("Ops schema:", OPS)
print("Raw volume path:", RAW_VOLUME)
print("Mode:", MODE)


Catalog: workspace
Bronze schema: workspace.ecommerce_hypermarketing_dev_bronze
Silver schema: workspace.ecommerce_hypermarketing_dev_silver
Gold schema: workspace.ecommerce_hypermarketing_dev_gold
Ops schema: workspace.ecommerce_hypermarketing_dev_ops
Raw volume path: /Volumes/workspace/ecommerce_hypermarketing_dev_bronze/ecommerce_raw
Mode: batch


In [0]:

# COMMAND ----------

# ============================================================
# 1. Imports and helpers
# ============================================================

from uuid import uuid4
from datetime import datetime, timedelta
import random

from pyspark.sql import functions as F
from pyspark.sql import Window
from delta.tables import DeltaTable

RUN_ID = str(uuid4())


In [0]:


def qname(schema_name: str, table_name: str) -> str:
    parts = schema_name.split(".")
    return ".".join([f"`{p}`" for p in parts] + [f"`{table_name}`"])


def table_exists(table_name: str) -> bool:
    try:
        spark.table(table_name).limit(1).count()
        return True
    except Exception:
        return False


def merge_upsert(df, target_table: str, keys):
    if not table_exists(target_table):
        df.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(target_table)
        return df.count()

    condition = " AND ".join([f"t.`{k}` = s.`{k}`" for k in keys])
    delta = DeltaTable.forName(spark, target_table)
    (
        delta.alias("t")
        .merge(df.alias("s"), condition)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    return df.count()


def audit(task_name, table_name, status, row_count=None, message=None):
    rows = [(RUN_ID, task_name, table_name, status, row_count, message, datetime.utcnow(), datetime.utcnow())]
    df = spark.createDataFrame(rows, "run_id string, task_name string, table_name string, status string, row_count long, message string, started_at timestamp, ended_at timestamp")
    df.write.format("delta").mode("append").saveAsTable(qname(OPS, "etl_audit_log"))


def dq_result(layer, table_name, rule_name, rule_status, failed_count):
    rows = [(RUN_ID, layer, table_name, rule_name, rule_status, int(failed_count), datetime.utcnow())]
    df = spark.createDataFrame(rows, "run_id string, layer string, table_name string, rule_name string, rule_status string, failed_count long, checked_at timestamp")
    df.write.format("delta").mode("append").saveAsTable(qname(OPS, "data_quality_results"))


def normalize_columns(df):
    for col_name in df.columns:
        normalized = col_name.strip().lower().replace(" ", "_").replace("-", "_")
        if col_name != normalized:
            df = df.withColumnRenamed(col_name, normalized)
    return df


def read_csv(path):
    return (
        spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .option("multiLine", True)
        .option("escape", '"')
        .load(path)
    )

In [0]:


# COMMAND ----------

# ============================================================
# 2. Metadata registry
# ============================================================

TABLE_REGISTRY = {
    "product_master": {
        "source_file": "product_master.csv",
        "bronze": "raw_product_master",
        "silver": "dim_product",
        "keys": ["sku"],
        "columns": {
            "sku": "string",
            "product_name": "string",
            "category": "string",
            "brand": "string",
            "unit_price": "decimal(12,2)",
            "margin_pct": "decimal(6,2)",
            "base_stock": "int",
            "initial_stock_cover_hrs": "decimal(8,2)",
        },
    },
    "marketing_performance_hourly": {
        "source_file": "marketing_performance_hourly.csv",
        "bronze": "raw_marketing_performance_hourly",
        "silver": "fact_marketing_performance_hourly",
        "keys": ["hour_key", "sku"],
        "columns": {
            "timestamp": "timestamp",
            "hour_key": "string",
            "sku": "string",
            "product_name": "string",
            "category": "string",
            "sessions": "int",
            "clicks": "int",
            "ctr_pct": "decimal(8,2)",
            "orders": "int",
            "units_sold": "int",
            "cvr_pct": "decimal(8,2)",
            "gross_revenue": "decimal(14,2)",
            "ad_spend": "decimal(14,2)",
            "roas": "decimal(8,2)",
            "demand_index": "decimal(8,2)",
            "demand_spike_pct": "decimal(8,2)",
            "discount_pct": "decimal(6,2)",
            "avg_dwell_time_sec": "int",
            "push_score": "decimal(8,2)",
        },
    },
    "stock_movement_hourly": {
        "source_file": "stock_movement_hourly.csv",
        "bronze": "raw_stock_movement_hourly",
        "silver": "fact_stock_movement_hourly",
        "keys": ["hour_key", "sku"],
        "columns": {
            "timestamp": "timestamp",
            "hour_key": "string",
            "sku": "string",
            "product_name": "string",
            "category": "string",
            "opening_stock": "int",
            "units_sold": "int",
            "restocked_units": "int",
            "closing_stock": "int",
            "stock_cover_hrs": "decimal(8,2)",
            "stock_risk_flag": "string",
        },
    },
    "order_transactions": {
        "source_file": "order_transactions_sample.csv",
        "bronze": "raw_order_transactions",
        "silver": "fact_order_transactions",
        "keys": ["order_id"],
        "columns": {
            "order_id": "string",
            "order_timestamp": "timestamp",
            "sku": "string",
            "product_name": "string",
            "category": "string",
            "quantity": "int",
            "order_value": "decimal(14,2)",
            "discount_pct": "decimal(6,2)",
            "traffic_channel": "string",
            "city": "string",
            "customer_type": "string",
            "payment_mode": "string",
        },
    },
    "orders_hourly_summary": {
        "source_file": "orders_hourly_summary.csv",
        "bronze": "raw_orders_hourly_summary",
        "silver": "fact_orders_hourly_summary",
        "keys": ["hour_key"],
        "columns": {
            "timestamp": "timestamp",
            "hour_key": "string",
            "total_orders": "int",
            "total_units": "int",
            "total_sessions": "int",
            "revenue": "decimal(14,2)",
            "ad_spend": "decimal(14,2)",
            "conversion_rate_pct": "decimal(8,2)",
            "aov": "decimal(12,2)",
        },
    },
    "category_performance_hourly": {
        "source_file": "category_performance_hourly.csv",
        "bronze": "raw_category_performance_hourly",
        "silver": "fact_category_performance_hourly",
        "keys": ["hour_key", "category"],
        "columns": {
            "timestamp": "timestamp",
            "hour_key": "string",
            "category": "string",
            "sessions": "int",
            "orders": "int",
            "revenue": "decimal(14,2)",
            "ad_spend": "decimal(14,2)",
            "avg_demand_spike_pct": "decimal(8,2)",
            "avg_push_score": "decimal(8,2)",
            "cvr_pct": "decimal(8,2)",
            "roas": "decimal(8,2)",
        },
    },
    "channel_performance_hourly": {
        "source_file": "channel_performance_hourly.csv",
        "bronze": "raw_channel_performance_hourly",
        "silver": "fact_channel_performance_hourly",
        "keys": ["hour_key", "channel"],
        "columns": {
            "timestamp": "timestamp",
            "hour_key": "string",
            "channel": "string",
            "sessions": "int",
            "orders": "int",
            "revenue": "decimal(14,2)",
            "ad_spend": "decimal(14,2)",
            "cvr_pct": "decimal(8,2)",
            "roas": "decimal(8,2)",
        },
    },
    "restock_schedule": {
        "source_file": "restock_schedule.csv",
        "bronze": "raw_restock_schedule",
        "silver": "fact_restock_schedule",
        "keys": ["sku", "scheduled_restock_timestamp"],
        "columns": {
            "sku": "string",
            "product_name": "string",
            "scheduled_restock_timestamp": "timestamp",
            "restock_units": "int",
            "status": "string",
        },
    },
}

In [0]:
# ============================================================
# Create schemas under an existing catalog
# IMPORTANT:
# Do not create catalog here unless your metastore has a storage root
# or you provide a catalog MANAGED LOCATION.
# Use existing catalog, usually: workspace
# ============================================================

CATALOG = "hackathon"  # Use an existing catalog
SCHEMA_PREFIX = "ecommerce_hypermarketing_dev"

BRONZE = f"{CATALOG}.{SCHEMA_PREFIX}_bronze"
SILVER = f"{CATALOG}.{SCHEMA_PREFIX}_silver"
GOLD = f"{CATALOG}.{SCHEMA_PREFIX}_gold"
OPS = f"{CATALOG}.{SCHEMA_PREFIX}_ops"

# Optional validation: confirm catalog exists
catalogs = [row.catalog for row in spark.sql("SHOW CATALOGS").collect()]
if CATALOG not in catalogs:
    raise ValueError(
        f"Catalog '{CATALOG}' does not exist. "
        f"Create it from Catalog Explorer UI using Default Storage, "
        f"or ask your workspace admin to create it with a managed location."
    )

# Create schemas only
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA_PREFIX}_bronze`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA_PREFIX}_silver`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA_PREFIX}_gold`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA_PREFIX}_ops`")

print("Schemas created successfully.")

Schemas created successfully.


In [0]:

# COMMAND ----------

# ============================================================
# 3. Create UC schemas, volumes, and ops tables
# ============================================================



spark.sql(f"CREATE VOLUME IF NOT EXISTS `{CATALOG}`.`{SCHEMA_PREFIX}_bronze`.`ecommerce_raw`")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{CATALOG}`.`{SCHEMA_PREFIX}_bronze`.`ecommerce_checkpoint`")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{CATALOG}`.`{SCHEMA_PREFIX}_bronze`.`ecommerce_schema`")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {qname(OPS, 'etl_audit_log')} (
  run_id STRING,
  task_name STRING,
  table_name STRING,
  status STRING,
  row_count BIGINT,
  message STRING,
  started_at TIMESTAMP,
  ended_at TIMESTAMP
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {qname(OPS, 'data_quality_results')} (
  run_id STRING,
  layer STRING,
  table_name STRING,
  rule_name STRING,
  rule_status STRING,
  failed_count BIGINT,
  checked_at TIMESTAMP
) USING DELTA
""")

print("UC schemas, volumes, and ops tables created.")
print("Upload CSV files here if running batch mode:", RAW_VOLUME)


UC schemas, volumes, and ops tables created.
Upload CSV files here if running batch mode: /Volumes/workspace/ecommerce_hypermarketing_dev_bronze/ecommerce_raw


In [0]:

# COMMAND ----------

# ============================================================
# 4. Optional: generate sample data directly into raw volume
# ============================================================

if MODE == "generate_sample_data":
    import pandas as pd
    import numpy as np

    np.random.seed(42)
    current_hour = pd.Timestamp.now().floor("H")
    hours = pd.date_range(end=current_hour, periods=48, freq="H")

    products = pd.DataFrame({
        "SKU": ["EL-4821", "FA-1108", "BE-2044", "HM-3175", "SP-1451", "GR-5632", "EL-5540", "FA-2213"],
        "Product_Name": ["Wireless Earbuds", "Summer Linen Shirt", "Vitamin C Serum", "Air Fryer 4L", "Yoga Mat Pro", "Healthy Snack Box", "Smartwatch Lite", "Running Shorts"],
        "Category": ["Electronics", "Fashion", "Beauty", "Home", "Sports", "Grocery", "Electronics", "Fashion"],
        "Brand": ["SoundPeak", "UrbanWeave", "GlowLab", "HomeEase", "FlexFit", "SnackWell", "PulseTime", "SprintWear"],
        "Unit_Price": [79, 34, 24, 129, 32, 19, 119, 28],
        "Margin_Pct": [38, 44, 57, 31, 46, 29, 34, 42],
        "Base_Stock": [1800, 2400, 2600, 900, 1700, 3200, 1200, 2100],
        "Initial_Stock_Cover_Hrs": [9, 14, 18, 22, 19, 31, 16, 20],
    })

    settings = {
        "EL-4821": (1500, 4.8, 5.1, 38, 6.2),
        "FA-1108": (1300, 3.9, 4.2, 26, 4.7),
        "BE-2044": (1200, 4.2, 4.8, 24, 5.4),
        "HM-3175": (700, 2.9, 2.7, 20, 3.1),
        "SP-1451": (900, 3.4, 3.3, 18, 4.0),
        "GR-5632": (650, 2.1, 2.3, 7, 2.2),
        "EL-5540": (1000, 4.5, 4.7, 28, 5.7),
        "FA-2213": (850, 3.0, 3.5, 16, 3.8),
    }

    pattern = np.array([0.72,0.68,0.64,0.61,0.63,0.71,0.84,0.96,1.04,1.10,1.15,1.20,1.24,1.27,1.31,1.36,1.43,1.49,1.55,1.49,1.38,1.24,1.08,0.92])
    demand_index = np.clip(100 * pattern[[h.hour for h in hours]] * np.linspace(0.95, 1.08, len(hours)) + np.random.normal(0, 3.5, len(hours)), 52, None)

    perf_rows, stock_rows, kpi_rows, cat_rows, channel_rows, tx_rows = [], [], [], [], [], []
    current_stock = {r.SKU: int(r.Base_Stock) for r in products.itertuples()}
    order_counter = 100000

    for ts, di in zip(hours, demand_index):
        hour_rev = hour_orders = hour_units = hour_sessions = hour_ad = 0
        for r in products.itertuples():
            base_sessions, ctr, cvr, spike, roas_target = settings[r.SKU]
            sessions = int(base_sessions * di / 100 * np.random.uniform(0.92, 1.08))
            ctr_pct = max(1, ctr + np.random.normal(0, 0.2))
            cvr_pct = max(1, cvr + np.random.normal(0, 0.18))
            clicks = int(sessions * ctr_pct / 100)
            orders = max(1, int(sessions * cvr_pct / 100))
            units = int(orders * np.random.uniform(1.15, 1.35))
            revenue = round(units * r.Unit_Price * np.random.uniform(0.96, 1.04), 2)
            ad_spend = round(revenue / max(1.5, roas_target + np.random.normal(0, 0.25)), 2)
            demand_spike = max(0, spike + np.random.normal(0, 2.2))
            opening = current_stock[r.SKU]
            restock = int(np.random.choice([0,0,0,0,300], p=[.82,.05,.05,.05,.03]))
            closing = max(0, opening - units + restock)
            current_stock[r.SKU] = closing
            cover = round(closing / max(units, 1), 1)
            risk = "High" if cover < 12 else "Medium" if cover < 18 else "Low"
            push_score = round(0.27*demand_spike + 0.18*roas_target*10 + 0.18*cvr_pct*10 + 0.17*r.Margin_Pct + 0.10*min(100, cover/24*100), 1)

            perf_rows.append([ts, ts.strftime("%Y%m%d%H"), r.SKU, r.Product_Name, r.Category, sessions, clicks, round(ctr_pct,2), orders, units, round(cvr_pct,2), revenue, ad_spend, round(revenue/ad_spend,2), round(float(di),1), round(demand_spike,1), round(np.random.uniform(0,8),1), int(np.random.uniform(40,90)), push_score])
            stock_rows.append([ts, ts.strftime("%Y%m%d%H"), r.SKU, r.Product_Name, r.Category, opening, units, restock, closing, cover, risk])
            hour_rev += revenue; hour_orders += orders; hour_units += units; hour_sessions += sessions; hour_ad += ad_spend

            for _ in range(min(8, max(1, orders//10))):
                order_counter += 1
                tx_rows.append([f"ORD{order_counter}", ts + pd.Timedelta(minutes=int(np.random.uniform(0,59))), r.SKU, r.Product_Name, r.Category, 1, round(r.Unit_Price*np.random.uniform(.95,1.05),2), round(np.random.uniform(0,10),1), np.random.choice(["Search","Social","App Push","Marketplace Ads","Display"]), np.random.choice(["Bangalore","Hyderabad","Chennai","Mumbai","Pune"]), np.random.choice(["New","Returning"]), np.random.choice(["UPI","Card","COD","Wallet"])])

        kpi_rows.append([ts, ts.strftime("%Y%m%d%H"), hour_orders, hour_units, hour_sessions, round(hour_rev,2), round(hour_ad,2), round(hour_orders/hour_sessions*100,2), round(hour_rev/hour_orders,2)])

    perf = pd.DataFrame(perf_rows, columns=["Timestamp","Hour_Key","SKU","Product_Name","Category","Sessions","Clicks","CTR_Pct","Orders","Units_Sold","CVR_Pct","Gross_Revenue","Ad_Spend","ROAS","Demand_Index","Demand_Spike_Pct","Discount_Pct","Avg_Dwell_Time_Sec","Push_Score"])
    stock = pd.DataFrame(stock_rows, columns=["Timestamp","Hour_Key","SKU","Product_Name","Category","Opening_Stock","Units_Sold","Restocked_Units","Closing_Stock","Stock_Cover_Hrs","Stock_Risk_Flag"])
    orders_hourly = pd.DataFrame(kpi_rows, columns=["Timestamp","Hour_Key","Total_Orders","Total_Units","Total_Sessions","Revenue","Ad_Spend","Conversion_Rate_Pct","AOV"])
    tx = pd.DataFrame(tx_rows, columns=["Order_ID","Order_Timestamp","SKU","Product_Name","Category","Quantity","Order_Value","Discount_Pct","Traffic_Channel","City","Customer_Type","Payment_Mode"])
    cat = perf.groupby(["Timestamp","Hour_Key","Category"], as_index=False).agg(Sessions=("Sessions","sum"), Orders=("Orders","sum"), Revenue=("Gross_Revenue","sum"), Ad_Spend=("Ad_Spend","sum"), Avg_Demand_Spike_Pct=("Demand_Spike_Pct","mean"), Avg_Push_Score=("Push_Score","mean"))
    cat["CVR_Pct"] = (cat["Orders"]/cat["Sessions"]*100).round(2)
    cat["ROAS"] = (cat["Revenue"]/cat["Ad_Spend"]).round(2)
    channels = []
    for _, row in orders_hourly.iterrows():
        for ch, mix, boost in [("Search",.30,1.1),("Social",.24,.95),("App Push",.16,1.2),("Marketplace Ads",.18,.9),("Display",.12,.75)]:
            rev = row.Revenue*mix*boost*np.random.uniform(.92,1.08)
            spend = rev/np.random.uniform(2.5,6.5)
            channels.append([row.Timestamp,row.Hour_Key,ch,int(row.Total_Sessions*mix),int(row.Total_Orders*mix*boost),round(rev,2),round(spend,2),round(row.Conversion_Rate_Pct*boost,2),round(rev/spend,2)])
    channel = pd.DataFrame(channels, columns=["Timestamp","Hour_Key","Channel","Sessions","Orders","Revenue","Ad_Spend","CVR_Pct","ROAS"])
    restock = pd.DataFrame({"SKU": products.SKU, "Product_Name": products.Product_Name, "Scheduled_Restock_Timestamp": current_hour + pd.to_timedelta(np.arange(len(products))+1, unit="h"), "Restock_Units": [300,400,500,200,350,600,280,260], "Status": "Planned"})

    local_dir = "/tmp/ecommerce_hypermarketing_seed"
    dbutils.fs.rm("file:" + local_dir, True)
    dbutils.fs.mkdirs("file:" + local_dir)

    outputs = {
        "product_master.csv": products,
        "marketing_performance_hourly.csv": perf,
        "stock_movement_hourly.csv": stock,
        "order_transactions_sample.csv": tx,
        "orders_hourly_summary.csv": orders_hourly,
        "category_performance_hourly.csv": cat,
        "channel_performance_hourly.csv": channel,
        "restock_schedule.csv": restock,
    }
    for name, df in outputs.items():
        df.to_csv(f"{local_dir}/{name}", index=False)

    dbutils.fs.cp("file:" + local_dir, RAW_VOLUME, recurse=True)
    print("Generated and uploaded sample data to", RAW_VOLUME)
else:
    print("Sample generation skipped. Expecting CSV files in", RAW_VOLUME)


Sample generation skipped. Expecting CSV files in /Volumes/workspace/ecommerce_hypermarketing_dev_bronze/ecommerce_raw


In [0]:
# ============================================================
# Safe catalog/schema setup
# ============================================================

# 1. Pick an existing Unity Catalog catalog.
# Run SHOW CATALOGS first and set this value to one of the returned catalogs.
CATALOG = "hackathon"   # Change this if your catalog name is different

SCHEMA_PREFIX = "ecommerce_hypermarketing_dev"

# 2. Validate catalog exists
available_catalogs = [row.catalog for row in spark.sql("SHOW CATALOGS").collect()]

print("Available catalogs:", available_catalogs)

if CATALOG not in available_catalogs:
    raise ValueError(
        f"Catalog '{CATALOG}' does not exist. "
        f"Available catalogs are: {available_catalogs}. "
        "Set CATALOG to one of the available Unity Catalog catalogs."
    )

# 3. Define schemas
BRONZE = f"{CATALOG}.{SCHEMA_PREFIX}_bronze"
SILVER = f"{CATALOG}.{SCHEMA_PREFIX}_silver"
GOLD = f"{CATALOG}.{SCHEMA_PREFIX}_gold"
OPS = f"{CATALOG}.{SCHEMA_PREFIX}_ops"

# 4. Create schemas under the existing catalog
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA_PREFIX}_bronze`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA_PREFIX}_silver`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA_PREFIX}_gold`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA_PREFIX}_ops`")

# 5. Create volumes under Bronze schema
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{CATALOG}`.`{SCHEMA_PREFIX}_bronze`.`ecommerce_raw`")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{CATALOG}`.`{SCHEMA_PREFIX}_bronze`.`ecommerce_checkpoint`")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{CATALOG}`.`{SCHEMA_PREFIX}_bronze`.`ecommerce_schema`")

RAW_VOLUME = f"/Volumes/{CATALOG}/{SCHEMA_PREFIX}_bronze/ecommerce_raw"
CHECKPOINT_VOLUME = f"/Volumes/{CATALOG}/{SCHEMA_PREFIX}_bronze/ecommerce_checkpoint"
SCHEMA_VOLUME = f"/Volumes/{CATALOG}/{SCHEMA_PREFIX}_bronze/ecommerce_schema"

print("Schemas and volumes created successfully.")
print("Bronze schema:", BRONZE)
print("Silver schema:", SILVER)
print("Gold schema:", GOLD)
print("Ops schema:", OPS)
print("Raw volume path:", RAW_VOLUME)

Available catalogs: ['hackathon', 'hive_metastore', 'jumia_catalog', 'samples', 'system']
Schemas and volumes created successfully.
Bronze schema: hackathon.ecommerce_hypermarketing_dev_bronze
Silver schema: hackathon.ecommerce_hypermarketing_dev_silver
Gold schema: hackathon.ecommerce_hypermarketing_dev_gold
Ops schema: hackathon.ecommerce_hypermarketing_dev_ops
Raw volume path: /Volumes/hackathon/ecommerce_hypermarketing_dev_bronze/ecommerce_raw


In [0]:
# ============================================================
# Create product_master.csv in Unity Catalog Volume
# Path: /Volumes/hackathon/ecommerce_hypermarketing_dev_bronze/ecommerce_raw/product_master.csv
# ============================================================

product_master_csv = """SKU,Product_Name,Category,Brand,Unit_Price,Margin_Pct,Base_Stock,Initial_Stock_Cover_Hrs
EL-4821,Wireless Earbuds,Electronics,SoundPeak,79,38,1800,9
FA-1108,Summer Linen Shirt,Fashion,UrbanWeave,34,44,2400,14
BE-2044,Vitamin C Serum,Beauty,GlowLab,24,57,2600,18
HM-3175,Air Fryer 4L,Home,HomeEase,129,31,900,22
SP-1451,Yoga Mat Pro,Sports,FlexFit,32,46,1700,19
GR-5632,Healthy Snack Box,Grocery,SnackWell,19,29,3200,31
EL-5540,Smartwatch Lite,Electronics,PulseTime,119,34,1200,16
FA-2213,Running Shorts,Fashion,SprintWear,28,42,2100,20
"""

target_path = "/Volumes/hackathon/ecommerce_hypermarketing_dev_bronze/ecommerce_raw/product_master.csv"

dbutils.fs.put(target_path, product_master_csv, overwrite=True)

print(f"File written successfully to: {target_path}")

Wrote 536 bytes.
File written successfully to: /Volumes/hackathon/ecommerce_hypermarketing_dev_bronze/ecommerce_raw/product_master.csv


In [0]:
# ============================================================
# Create product_master.csv in UC Volume
# ============================================================

product_master_csv = """SKU,Product_Name,Category,Brand,Unit_Price,Margin_Pct,Base_Stock,Initial_Stock_Cover_Hrs
EL-4821,Wireless Earbuds,Electronics,SoundPeak,79.00,38.00,1800,9.00
FA-1108,Summer Linen Shirt,Fashion,UrbanWeave,34.00,44.00,2400,14.00
BE-2044,Vitamin C Serum,Beauty,GlowLab,24.00,57.00,2600,18.00
HM-3175,Air Fryer 4L,Home,HomeEase,129.00,31.00,900,22.00
SP-1451,Yoga Mat Pro,Sports,FlexFit,32.00,46.00,1700,19.00
GR-5632,Healthy Snack Box,Grocery,SnackWell,19.00,29.00,3200,31.00
EL-5540,Smartwatch Lite,Electronics,PulseTime,119.00,34.00,1200,16.00
FA-2213,Running Shorts,Fashion,SprintWear,28.00,42.00,2100,20.00
"""

target_path = "/Volumes/hackathon/ecommerce_hypermarketing_dev_bronze/ecommerce_raw/product_master.csv"

dbutils.fs.put(target_path, product_master_csv, overwrite=True)

print(f"product_master.csv created successfully at: {target_path}")


Wrote 608 bytes.
product_master.csv created successfully at: /Volumes/hackathon/ecommerce_hypermarketing_dev_bronze/ecommerce_raw/product_master.csv


In [0]:
# ============================================================
# Create marketing_performance_hourly.csv in UC Volume
# Target:
# /Volumes/hackathon/ecommerce_hypermarketing_dev_bronze/ecommerce_raw/marketing_performance_hourly.csv
# ============================================================

import pandas as pd
import numpy as np

np.random.seed(42)

target_path = "/Volumes/hackathon/ecommerce_hypermarketing_dev_bronze/ecommerce_raw/marketing_performance_hourly.csv"

# Generate 48 hourly records per product
current_hour = pd.Timestamp.now().floor("H")
hours = pd.date_range(end=current_hour, periods=48, freq="H")

products = pd.DataFrame({
    "SKU": [
        "EL-4821", "FA-1108", "BE-2044", "HM-3175",
        "SP-1451", "GR-5632", "EL-5540", "FA-2213"
    ],
    "Product_Name": [
        "Wireless Earbuds", "Summer Linen Shirt", "Vitamin C Serum", "Air Fryer 4L",
        "Yoga Mat Pro", "Healthy Snack Box", "Smartwatch Lite", "Running Shorts"
    ],
    "Category": [
        "Electronics", "Fashion", "Beauty", "Home",
        "Sports", "Grocery", "Electronics", "Fashion"
    ],
    "Unit_Price": [79, 34, 24, 129, 32, 19, 119, 28],
    "Margin_Pct": [38, 44, 57, 31, 46, 29, 34, 42]
})

product_settings = {
    "EL-4821": {"base_sessions": 1500, "ctr": 4.8, "cvr": 5.1, "demand_spike": 38, "roas_target": 6.2},
    "FA-1108": {"base_sessions": 1300, "ctr": 3.9, "cvr": 4.2, "demand_spike": 26, "roas_target": 4.7},
    "BE-2044": {"base_sessions": 1200, "ctr": 4.2, "cvr": 4.8, "demand_spike": 24, "roas_target": 5.4},
    "HM-3175": {"base_sessions": 700,  "ctr": 2.9, "cvr": 2.7, "demand_spike": 20, "roas_target": 3.1},
    "SP-1451": {"base_sessions": 900,  "ctr": 3.4, "cvr": 3.3, "demand_spike": 18, "roas_target": 4.0},
    "GR-5632": {"base_sessions": 650,  "ctr": 2.1, "cvr": 2.3, "demand_spike": 7,  "roas_target": 2.2},
    "EL-5540": {"base_sessions": 1000, "ctr": 4.5, "cvr": 4.7, "demand_spike": 28, "roas_target": 5.7},
    "FA-2213": {"base_sessions": 850,  "ctr": 3.0, "cvr": 3.5, "demand_spike": 16, "roas_target": 3.8},
}

# Intraday demand curve
intraday_pattern = np.array([
    0.72, 0.68, 0.64, 0.61, 0.63, 0.71,
    0.84, 0.96, 1.04, 1.10, 1.15, 1.20,
    1.24, 1.27, 1.31, 1.36, 1.43, 1.49,
    1.55, 1.49, 1.38, 1.24, 1.08, 0.92
])

rows = []

for ts in hours:
    hour_factor = intraday_pattern[ts.hour]

    total_range_seconds = max(
        1,
        (hours.max() - hours.min()).total_seconds()
    )

    trend_factor = 1.0 + (
        (ts - hours.min()).total_seconds() / total_range_seconds
    ) * 0.08

    base_demand_index = max(
        52,
        100 * hour_factor * trend_factor + np.random.normal(0, 3.5)
    )

    for product in products.itertuples(index=False):
        settings = product_settings[product.SKU]

        category_effect = {
            "Electronics": 1.10,
            "Beauty": 1.05,
            "Fashion": 1.00,
            "Home": 0.86,
            "Sports": 0.92,
            "Grocery": 0.76
        }[product.Category]

        sessions = max(
            50,
            int(
                settings["base_sessions"]
                * (base_demand_index / 100)
                * category_effect
                * np.random.uniform(0.92, 1.08)
            )
        )

        ctr_pct = max(
            1.0,
            settings["ctr"] + np.random.normal(0, 0.18)
        )

        cvr_pct = max(
            1.0,
            settings["cvr"] + np.random.normal(0, 0.16)
        )

        clicks = int(round(sessions * ctr_pct / 100))
        orders = max(1, int(round(sessions * cvr_pct / 100)))
        units_sold = max(1, int(round(orders * np.random.uniform(1.15, 1.35))))

        gross_revenue = round(
            units_sold * product.Unit_Price * np.random.uniform(0.96, 1.04),
            2
        )

        roas_target = max(
            1.5,
            settings["roas_target"] + np.random.normal(0, 0.25)
        )

        ad_spend = round(gross_revenue / roas_target, 2)
        roas = round(gross_revenue / ad_spend, 2) if ad_spend else 0

        demand_spike_pct = max(
            0,
            settings["demand_spike"] + np.random.normal(0, 2.2)
        )

        discount_pct = round(
            np.clip(
                np.random.normal(
                    5 if product.Category in ["Electronics", "Fashion"] else 3.5,
                    1.8
                ),
                0,
                12
            ),
            1
        )

        avg_dwell_time_sec = int(
            round(
                np.clip(
                    np.random.normal(60, 10),
                    25,
                    120
                )
            )
        )

        push_score = round(
            0.27 * demand_spike_pct +
            0.18 * roas * 10 +
            0.18 * cvr_pct * 10 +
            0.17 * product.Margin_Pct +
            0.10 * min(100, ctr_pct * 12),
            1
        )

        rows.append({
            "Timestamp": ts,
            "Hour_Key": ts.strftime("%Y%m%d%H"),
            "SKU": product.SKU,
            "Product_Name": product.Product_Name,
            "Category": product.Category,
            "Sessions": sessions,
            "Clicks": clicks,
            "CTR_Pct": round(ctr_pct, 2),
            "Orders": orders,
            "Units_Sold": units_sold,
            "CVR_Pct": round(cvr_pct, 2),
            "Gross_Revenue": gross_revenue,
            "Ad_Spend": ad_spend,
            "ROAS": roas,
            "Demand_Index": round(base_demand_index, 1),
            "Demand_Spike_Pct": round(demand_spike_pct, 1),
            "Discount_Pct": discount_pct,
            "Avg_Dwell_Time_Sec": avg_dwell_time_sec,
            "Push_Score": push_score
        })

marketing_df = pd.DataFrame(rows)

# IMPORTANT:
# Do not use pandas to_csv directly on /Volumes path.
# Convert dataframe to CSV text and write using dbutils.fs.put.
csv_data = marketing_df.to_csv(index=False)

dbutils.fs.put(
    target_path,
    csv_data,
    overwrite=True
)

print(f"marketing_performance_hourly.csv created successfully at: {target_path}")
print(f"Rows written: {len(marketing_df)}")

/home/spark-75af134b-bc6f-47f2-bacf-d8/.ipykernel/2076/command-8778560173977076-966478287:15: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  current_hour = pd.Timestamp.now().floor("H")
/home/spark-75af134b-bc6f-47f2-bacf-d8/.ipykernel/2076/command-8778560173977076-966478287:16: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hours = pd.date_range(end=current_hour, periods=48, freq="H")


Wrote 49777 bytes.
marketing_performance_hourly.csv created successfully at: /Volumes/hackathon/ecommerce_hypermarketing_dev_bronze/ecommerce_raw/marketing_performance_hourly.csv
Rows written: 384


In [0]:
# ============================================================
# Create stock_movement_hourly.csv directly in UC Volume
# Target:
# /Volumes/hackathon/ecommerce_hypermarketing_dev_bronze/ecommerce_raw/stock_movement_hourly.csv
# ============================================================

import pandas as pd
import numpy as np

np.random.seed(42)

# ------------------------------------------------------------
# Target catalog/schema/volume/file
# ------------------------------------------------------------

CATALOG = "hackathon"
BRONZE_SCHEMA = "ecommerce_hypermarketing_dev_bronze"
VOLUME = "ecommerce_raw"

RAW_VOLUME = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/{VOLUME}"
target_path = f"{RAW_VOLUME}/stock_movement_hourly.csv"

print("Target file path:", target_path)

# ------------------------------------------------------------
# Optional: create schema and volume if they do not exist
# Remove these lines if you do not have CREATE SCHEMA / CREATE VOLUME permissions
# ------------------------------------------------------------

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{BRONZE_SCHEMA}`")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{CATALOG}`.`{BRONZE_SCHEMA}`.`{VOLUME}`")

# ------------------------------------------------------------
# Product seed data
# ------------------------------------------------------------

current_hour = pd.Timestamp.now().floor("H")
hours = pd.date_range(end=current_hour, periods=48, freq="H")

products = pd.DataFrame({
    "SKU": [
        "EL-4821", "FA-1108", "BE-2044", "HM-3175",
        "SP-1451", "GR-5632", "EL-5540", "FA-2213"
    ],
    "Product_Name": [
        "Wireless Earbuds",
        "Summer Linen Shirt",
        "Vitamin C Serum",
        "Air Fryer 4L",
        "Yoga Mat Pro",
        "Healthy Snack Box",
        "Smartwatch Lite",
        "Running Shorts"
    ],
    "Category": [
        "Electronics",
        "Fashion",
        "Beauty",
        "Home",
        "Sports",
        "Grocery",
        "Electronics",
        "Fashion"
    ],
    "Base_Stock": [
        1800, 2400, 2600, 900,
        1700, 3200, 1200, 2100
    ],
    "Base_Units_Sold_Per_Hour": [
        125, 95, 75, 38,
        48, 32, 66, 44
    ]
})

# ------------------------------------------------------------
# Restock event plan
# Key = SKU
# Inner key = hour index in the 48-hour window
# Value = restocked units
# ------------------------------------------------------------

restock_plan = {
    "EL-4821": {9: 300, 38: 450},
    "FA-1108": {12: 400},
    "BE-2044": {20: 500},
    "HM-3175": {16: 200},
    "SP-1451": {36: 350},
    "GR-5632": {40: 600},
    "EL-5540": {18: 280},
    "FA-2213": {24: 260}
}

# ------------------------------------------------------------
# Intraday demand pattern
# Higher demand around afternoon/evening hours
# ------------------------------------------------------------

intraday_pattern = np.array([
    0.72, 0.68, 0.64, 0.61, 0.63, 0.71,
    0.84, 0.96, 1.04, 1.10, 1.15, 1.20,
    1.24, 1.27, 1.31, 1.36, 1.43, 1.49,
    1.55, 1.49, 1.38, 1.24, 1.08, 0.92
])

# ------------------------------------------------------------
# Generate stock movement rows
# ------------------------------------------------------------

rows = []

current_stock = {
    row.SKU: int(row.Base_Stock)
    for row in products.itertuples(index=False)
}

for hour_index, ts in enumerate(hours):
    hour_factor = intraday_pattern[ts.hour]

    total_range_seconds = max(
        1,
        (hours.max() - hours.min()).total_seconds()
    )

    trend_factor = 1.0 + (
        (ts - hours.min()).total_seconds() / total_range_seconds
    ) * 0.08

    for product in products.itertuples(index=False):
        sku = product.SKU

        opening_stock = int(current_stock[sku])

        category_effect = {
            "Electronics": 1.18,
            "Beauty": 1.08,
            "Fashion": 1.00,
            "Home": 0.78,
            "Sports": 0.86,
            "Grocery": 0.68
        }[product.Category]

        units_sold = int(
            max(
                1,
                round(
                    product.Base_Units_Sold_Per_Hour
                    * hour_factor
                    * trend_factor
                    * category_effect
                    * np.random.uniform(0.88, 1.12)
                )
            )
        )

        # Prevent negative stock
        units_sold = min(units_sold, opening_stock)

        restocked_units = restock_plan.get(sku, {}).get(hour_index, 0)

        closing_stock = max(
            0,
            opening_stock - units_sold + restocked_units
        )

        current_stock[sku] = closing_stock

        stock_cover_hrs = round(
            closing_stock / max(units_sold, 1),
            1
        )

        if stock_cover_hrs < 12:
            stock_risk_flag = "High"
        elif stock_cover_hrs < 18:
            stock_risk_flag = "Medium"
        else:
            stock_risk_flag = "Low"

        rows.append({
            "Timestamp": ts,
            "Hour_Key": ts.strftime("%Y%m%d%H"),
            "SKU": sku,
            "Product_Name": product.Product_Name,
            "Category": product.Category,
            "Opening_Stock": opening_stock,
            "Units_Sold": units_sold,
            "Restocked_Units": restocked_units,
            "Closing_Stock": closing_stock,
            "Stock_Cover_Hrs": stock_cover_hrs,
            "Stock_Risk_Flag": stock_risk_flag
        })

# ------------------------------------------------------------
# Convert to DataFrame
# ------------------------------------------------------------

stock_df = pd.DataFrame(rows)

# ------------------------------------------------------------
# IMPORTANT:
# Write directly to the UC Volume file using dbutils.fs.put.
# Do NOT use pandas.to_csv("/Volumes/...") directly.
# ------------------------------------------------------------

csv_data = stock_df.to_csv(index=False)

dbutils.fs.put(
    target_path,
    csv_data,
    overwrite=True
)

print("stock_movement_hourly.csv created successfully.")
print("File path:", target_path)
print("Rows written:", len(stock_df))

Target file path: /Volumes/hackathon/ecommerce_hypermarketing_dev_bronze/ecommerce_raw/stock_movement_hourly.csv


/home/spark-75af134b-bc6f-47f2-bacf-d8/.ipykernel/2076/command-8778560173977077-3390385465:37: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  current_hour = pd.Timestamp.now().floor("H")
/home/spark-75af134b-bc6f-47f2-bacf-d8/.ipykernel/2076/command-8778560173977077-3390385465:38: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hours = pd.date_range(end=current_hour, periods=48, freq="H")


Wrote 32757 bytes.
stock_movement_hourly.csv created successfully.
File path: /Volumes/hackathon/ecommerce_hypermarketing_dev_bronze/ecommerce_raw/stock_movement_hourly.csv
Rows written: 384


In [0]:
# ============================================================
# Create order_transactions_sample.csv directly in UC Volume
# Target:
# /Volumes/hackathon/ecommerce_hypermarketing_dev_bronze/ecommerce_raw/order_transactions_sample.csv
# ============================================================

import pandas as pd
import numpy as np

np.random.seed(42)

# ------------------------------------------------------------
# Target catalog/schema/volume/file
# ------------------------------------------------------------

CATALOG = "hackathon"
BRONZE_SCHEMA = "ecommerce_hypermarketing_dev_bronze"
VOLUME = "ecommerce_raw"

RAW_VOLUME = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/{VOLUME}"
target_path = f"{RAW_VOLUME}/order_transactions_sample.csv"

print("Target file path:", target_path)

# ------------------------------------------------------------
# Optional: create schema and volume if they do not exist
# Remove these lines if you do not have CREATE SCHEMA / CREATE VOLUME permissions
# ------------------------------------------------------------

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{BRONZE_SCHEMA}`")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{CATALOG}`.`{BRONZE_SCHEMA}`.`{VOLUME}`")

# ------------------------------------------------------------
# Product seed data
# ------------------------------------------------------------

products = pd.DataFrame({
    "SKU": [
        "EL-4821", "FA-1108", "BE-2044", "HM-3175",
        "SP-1451", "GR-5632", "EL-5540", "FA-2213"
    ],
    "Product_Name": [
        "Wireless Earbuds",
        "Summer Linen Shirt",
        "Vitamin C Serum",
        "Air Fryer 4L",
        "Yoga Mat Pro",
        "Healthy Snack Box",
        "Smartwatch Lite",
        "Running Shorts"
    ],
    "Category": [
        "Electronics",
        "Fashion",
        "Beauty",
        "Home",
        "Sports",
        "Grocery",
        "Electronics",
        "Fashion"
    ],
    "Unit_Price": [
        79.00, 34.00, 24.00, 129.00,
        32.00, 19.00, 119.00, 28.00
    ]
})

# ------------------------------------------------------------
# Transaction generation settings
# ------------------------------------------------------------

current_hour = pd.Timestamp.now().floor("H")
hours = pd.date_range(end=current_hour, periods=48, freq="H")

traffic_channels = [
    "Search",
    "Social",
    "App Push",
    "Marketplace Ads",
    "Display"
]

traffic_channel_prob = [
    0.30,
    0.24,
    0.18,
    0.18,
    0.10
]

cities = [
    "Bangalore",
    "Hyderabad",
    "Chennai",
    "Mumbai",
    "Pune"
]

city_prob = [
    0.35,
    0.22,
    0.16,
    0.16,
    0.11
]

customer_types = [
    "New",
    "Returning"
]

customer_type_prob = [
    0.38,
    0.62
]

payment_modes = [
    "UPI",
    "Card",
    "COD",
    "Wallet"
]

payment_mode_prob = [
    0.44,
    0.31,
    0.13,
    0.12
]

# Product-level order intensity
product_order_intensity = {
    "EL-4821": 1.35,
    "FA-1108": 1.10,
    "BE-2044": 1.15,
    "HM-3175": 0.65,
    "SP-1451": 0.78,
    "GR-5632": 0.55,
    "EL-5540": 1.05,
    "FA-2213": 0.75
}

# Intraday transaction curve
intraday_pattern = np.array([
    0.72, 0.68, 0.64, 0.61, 0.63, 0.71,
    0.84, 0.96, 1.04, 1.10, 1.15, 1.20,
    1.24, 1.27, 1.31, 1.36, 1.43, 1.49,
    1.55, 1.49, 1.38, 1.24, 1.08, 0.92
])

# ------------------------------------------------------------
# Generate order transaction rows
# ------------------------------------------------------------

rows = []
order_counter = 100000

for ts in hours:
    hour_factor = intraday_pattern[ts.hour]

    total_range_seconds = max(
        1,
        (hours.max() - hours.min()).total_seconds()
    )

    trend_factor = 1.0 + (
        (ts - hours.min()).total_seconds() / total_range_seconds
    ) * 0.08

    for product in products.itertuples(index=False):
        sku = product.SKU

        # Estimate transaction count per SKU per hour
        base_txn_count = int(
            max(
                2,
                round(
                    8
                    * product_order_intensity[sku]
                    * hour_factor
                    * trend_factor
                    * np.random.uniform(0.80, 1.25)
                )
            )
        )

        for _ in range(base_txn_count):
            order_counter += 1

            order_timestamp = ts + pd.Timedelta(
                minutes=int(np.random.uniform(0, 60)),
                seconds=int(np.random.uniform(0, 60))
            )

            quantity = int(
                max(
                    1,
                    round(np.random.choice([1, 1, 1, 2, 2, 3], p=[0.45, 0.22, 0.13, 0.12, 0.06, 0.02]))
                )
            )

            discount_pct = round(
                float(np.clip(np.random.normal(4.5, 1.7), 0, 12)),
                1
            )

            gross_value = product.Unit_Price * quantity
            discount_multiplier = 1 - (discount_pct / 100)

            order_value = round(
                gross_value * discount_multiplier * np.random.uniform(0.98, 1.02),
                2
            )

            rows.append({
                "Order_ID": f"ORD{order_counter}",
                "Order_Timestamp": order_timestamp,
                "SKU": product.SKU,
                "Product_Name": product.Product_Name,
                "Category": product.Category,
                "Quantity": quantity,
                "Order_Value": order_value,
                "Discount_Pct": discount_pct,
                "Traffic_Channel": np.random.choice(traffic_channels, p=traffic_channel_prob),
                "City": np.random.choice(cities, p=city_prob),
                "Customer_Type": np.random.choice(customer_types, p=customer_type_prob),
                "Payment_Mode": np.random.choice(payment_modes, p=payment_mode_prob)
            })

# ------------------------------------------------------------
# Convert to DataFrame
# ------------------------------------------------------------

orders_df = pd.DataFrame(rows)

# Sort for cleaner downstream processing
orders_df = orders_df.sort_values(
    ["Order_Timestamp", "SKU", "Order_ID"]
).reset_index(drop=True)

# ------------------------------------------------------------
# IMPORTANT:
# Write directly to the UC Volume file using dbutils.fs.put.
# Do NOT use pandas.to_csv("/Volumes/...") directly.
# ------------------------------------------------------------

csv_data = orders_df.to_csv(index=False)

dbutils.fs.put(
    target_path,
    csv_data,
    overwrite=True
)

print("order_transactions_sample.csv created successfully.")
print("File path:", target_path)
print("Rows written:", len(orders_df))

Target file path: /Volumes/hackathon/ecommerce_hypermarketing_dev_bronze/ecommerce_raw/order_transactions_sample.csv


/home/spark-75af134b-bc6f-47f2-bacf-d8/.ipykernel/2076/command-8778560173977078-4175282170:72: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  current_hour = pd.Timestamp.now().floor("H")
/home/spark-75af134b-bc6f-47f2-bacf-d8/.ipykernel/2076/command-8778560173977078-4175282170:73: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hours = pd.date_range(end=current_hour, periods=48, freq="H")


Wrote 344391 bytes.
order_transactions_sample.csv created successfully.
File path: /Volumes/hackathon/ecommerce_hypermarketing_dev_bronze/ecommerce_raw/order_transactions_sample.csv
Rows written: 3276


In [0]:
# ============================================================
# Create orders_hourly_summary.csv directly in UC Volume
# Target:
# /Volumes/hackathon/ecommerce_hypermarketing_dev_bronze/ecommerce_raw/orders_hourly_summary.csv
# ============================================================

import pandas as pd
import numpy as np

np.random.seed(42)

# ------------------------------------------------------------
# Target catalog/schema/volume/file
# ------------------------------------------------------------

CATALOG = "hackathon"
BRONZE_SCHEMA = "ecommerce_hypermarketing_dev_bronze"
VOLUME = "ecommerce_raw"

RAW_VOLUME = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/{VOLUME}"
target_path = f"{RAW_VOLUME}/orders_hourly_summary.csv"

print("Target file path:", target_path)

# ------------------------------------------------------------
# Optional: create schema and volume if they do not exist
# Remove these lines if you do not have CREATE SCHEMA / CREATE VOLUME permissions
# ------------------------------------------------------------

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{BRONZE_SCHEMA}`")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{CATALOG}`.`{BRONZE_SCHEMA}`.`{VOLUME}`")

# ------------------------------------------------------------
# Generate 48 hourly records
# ------------------------------------------------------------

current_hour = pd.Timestamp.now().floor("H")
hours = pd.date_range(end=current_hour, periods=48, freq="H")

# Intraday demand curve
intraday_pattern = np.array([
    0.72, 0.68, 0.64, 0.61, 0.63, 0.71,
    0.84, 0.96, 1.04, 1.10, 1.15, 1.20,
    1.24, 1.27, 1.31, 1.36, 1.43, 1.49,
    1.55, 1.49, 1.38, 1.24, 1.08, 0.92
])

rows = []

for ts in hours:
    hour_factor = intraday_pattern[ts.hour]

    total_range_seconds = max(
        1,
        (hours.max() - hours.min()).total_seconds()
    )

    trend_factor = 1.0 + (
        (ts - hours.min()).total_seconds() / total_range_seconds
    ) * 0.08

    demand_factor = hour_factor * trend_factor * np.random.uniform(0.94, 1.08)

    total_sessions = int(
        max(
            1000,
            round(9000 * demand_factor)
        )
    )

    conversion_rate_pct = round(
        np.clip(
            np.random.normal(4.05 + (hour_factor - 1.0) * 0.6, 0.18),
            2.5,
            5.2
        ),
        2
    )

    total_orders = int(
        max(
            20,
            round(total_sessions * conversion_rate_pct / 100)
        )
    )

    units_per_order = np.random.uniform(1.18, 1.34)

    total_units = int(
        max(
            total_orders,
            round(total_orders * units_per_order)
        )
    )

    aov = round(
        np.clip(
            np.random.normal(54 + (hour_factor - 1.0) * 8, 5),
            34,
            82
        ),
        2
    )

    revenue = round(
        total_orders * aov,
        2
    )

    roas_target = np.clip(
        np.random.normal(4.9 + (hour_factor - 1.0) * 0.4, 0.35),
        2.5,
        6.5
    )

    ad_spend = round(
        revenue / roas_target,
        2
    )

    rows.append({
        "Timestamp": ts,
        "Hour_Key": ts.strftime("%Y%m%d%H"),
        "Total_Orders": total_orders,
        "Total_Units": total_units,
        "Total_Sessions": total_sessions,
        "Revenue": revenue,
        "Ad_Spend": ad_spend,
        "Conversion_Rate_Pct": conversion_rate_pct,
        "AOV": aov
    })

orders_hourly_df = pd.DataFrame(rows)

# ------------------------------------------------------------
# IMPORTANT:
# Write directly to the UC Volume file using dbutils.fs.put.
# Do NOT use pandas.to_csv("/Volumes/...") directly.
# ------------------------------------------------------------

csv_data = orders_hourly_df.to_csv(index=False)

dbutils.fs.put(
    target_path,
    csv_data,
    overwrite=True
)

print("orders_hourly_summary.csv created successfully.")
print("File path:", target_path)
print("Rows written:", len(orders_hourly_df))

Target file path: /Volumes/hackathon/ecommerce_hypermarketing_dev_bronze/ecommerce_raw/orders_hourly_summary.csv


/home/spark-75af134b-bc6f-47f2-bacf-d8/.ipykernel/2076/command-8778560173977079-3354228208:37: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  current_hour = pd.Timestamp.now().floor("H")
/home/spark-75af134b-bc6f-47f2-bacf-d8/.ipykernel/2076/command-8778560173977079-3354228208:38: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hours = pd.date_range(end=current_hour, periods=48, freq="H")


Wrote 3547 bytes.
orders_hourly_summary.csv created successfully.
File path: /Volumes/hackathon/ecommerce_hypermarketing_dev_bronze/ecommerce_raw/orders_hourly_summary.csv
Rows written: 48


In [0]:
# ============================================================
# Create category_performance_hourly.csv directly in UC Volume
# Target:
# /Volumes/hackathon/ecommerce_hypermarketing_dev_bronze/ecommerce_raw/category_performance_hourly.csv
# ============================================================

import pandas as pd
import numpy as np

np.random.seed(42)

# ------------------------------------------------------------
# Target catalog/schema/volume/file
# ------------------------------------------------------------

CATALOG = "hackathon"
BRONZE_SCHEMA = "ecommerce_hypermarketing_dev_bronze"
VOLUME = "ecommerce_raw"

RAW_VOLUME = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/{VOLUME}"
target_path = f"{RAW_VOLUME}/category_performance_hourly.csv"

print("Target file path:", target_path)

# ------------------------------------------------------------
# Optional: create schema and volume if they do not exist
# Remove these lines if you do not have CREATE SCHEMA / CREATE VOLUME permissions
# ------------------------------------------------------------

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{BRONZE_SCHEMA}`")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{CATALOG}`.`{BRONZE_SCHEMA}`.`{VOLUME}`")

# ------------------------------------------------------------
# Generate 48 hourly records per category
# ------------------------------------------------------------

current_hour = pd.Timestamp.now().floor("H")
hours = pd.date_range(end=current_hour, periods=48, freq="H")

categories = pd.DataFrame({
    "Category": [
        "Electronics",
        "Fashion",
        "Beauty",
        "Home",
        "Sports",
        "Grocery"
    ],
    "Base_Sessions": [
        2800,
        2300,
        1900,
        1200,
        1000,
        850
    ],
    "Base_CVR_Pct": [
        4.90,
        4.10,
        4.65,
        2.80,
        3.35,
        2.25
    ],
    "Base_ROAS": [
        5.90,
        4.55,
        5.30,
        3.15,
        4.00,
        2.35
    ],
    "Base_Demand_Spike_Pct": [
        31,
        22,
        24,
        14,
        18,
        9
    ],
    "Base_Push_Score": [
        51,
        44,
        48,
        34,
        39,
        27
    ],
    "Avg_Order_Value": [
        98,
        38,
        29,
        124,
        34,
        21
    ]
})

# Intraday demand curve
intraday_pattern = np.array([
    0.72, 0.68, 0.64, 0.61, 0.63, 0.71,
    0.84, 0.96, 1.04, 1.10, 1.15, 1.20,
    1.24, 1.27, 1.31, 1.36, 1.43, 1.49,
    1.55, 1.49, 1.38, 1.24, 1.08, 0.92
])

rows = []

for ts in hours:
    hour_factor = intraday_pattern[ts.hour]

    total_range_seconds = max(
        1,
        (hours.max() - hours.min()).total_seconds()
    )

    trend_factor = 1.0 + (
        (ts - hours.min()).total_seconds() / total_range_seconds
    ) * 0.08

    for category in categories.itertuples(index=False):
        category_name = category.Category

        # Category-specific response multiplier
        category_effect = {
            "Electronics": 1.12,
            "Fashion": 1.02,
            "Beauty": 1.08,
            "Home": 0.82,
            "Sports": 0.90,
            "Grocery": 0.72
        }[category_name]

        sessions = int(
            max(
                100,
                round(
                    category.Base_Sessions
                    * hour_factor
                    * trend_factor
                    * category_effect
                    * np.random.uniform(0.90, 1.10)
                )
            )
        )

        cvr_pct = round(
            float(
                np.clip(
                    np.random.normal(
                        category.Base_CVR_Pct + (hour_factor - 1.0) * 0.30,
                        0.18
                    ),
                    1.5,
                    6.5
                )
            ),
            2
        )

        orders = int(
            max(
                1,
                round(sessions * cvr_pct / 100)
            )
        )

        revenue = round(
            orders
            * category.Avg_Order_Value
            * np.random.uniform(0.94, 1.08),
            2
        )

        roas = round(
            float(
                np.clip(
                    np.random.normal(
                        category.Base_ROAS + (hour_factor - 1.0) * 0.25,
                        0.28
                    ),
                    1.5,
                    7.5
                )
            ),
            2
        )

        ad_spend = round(
            revenue / roas,
            2
        )

        avg_demand_spike_pct = round(
            float(
                max(
                    0,
                    category.Base_Demand_Spike_Pct
                    + (hour_factor - 1.0) * 8
                    + np.random.normal(0, 2.0)
                )
            ),
            1
        )

        avg_push_score = round(
            float(
                np.clip(
                    category.Base_Push_Score
                    + (avg_demand_spike_pct * 0.18)
                    + (roas * 0.9)
                    + np.random.normal(0, 1.5),
                    10,
                    80
                )
            ),
            1
        )

        rows.append({
            "Timestamp": ts,
            "Hour_Key": ts.strftime("%Y%m%d%H"),
            "Category": category_name,
            "Sessions": sessions,
            "Orders": orders,
            "Revenue": revenue,
            "Ad_Spend": ad_spend,
            "Avg_Demand_Spike_Pct": avg_demand_spike_pct,
            "Avg_Push_Score": avg_push_score,
            "CVR_Pct": cvr_pct,
            "ROAS": roas
        })

category_perf_df = pd.DataFrame(rows)

# ------------------------------------------------------------
# IMPORTANT:
# Write directly to the UC Volume file using dbutils.fs.put.
# Do NOT use pandas.to_csv("/Volumes/...") directly.
# ------------------------------------------------------------

csv_data = category_perf_df.to_csv(index=False)

dbutils.fs.put(
    target_path,
    csv_data,
    overwrite=True
)

print("category_performance_hourly.csv created successfully.")
print("File path:", target_path)
print("Rows written:", len(category_perf_df))

Target file path: /Volumes/hackathon/ecommerce_hypermarketing_dev_bronze/ecommerce_raw/category_performance_hourly.csv


/home/spark-75af134b-bc6f-47f2-bacf-d8/.ipykernel/2076/command-8778560173977080-319483764:37: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  current_hour = pd.Timestamp.now().floor("H")
/home/spark-75af134b-bc6f-47f2-bacf-d8/.ipykernel/2076/command-8778560173977080-319483764:38: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hours = pd.date_range(end=current_hour, periods=48, freq="H")


Wrote 23636 bytes.
category_performance_hourly.csv created successfully.
File path: /Volumes/hackathon/ecommerce_hypermarketing_dev_bronze/ecommerce_raw/category_performance_hourly.csv
Rows written: 288


In [0]:
# ============================================================
# Generate ALL source CSV files for Bronze ingestion
# Target folder:
# /Volumes/hackathon/ecommerce_hypermarketing_dev_bronze/ecommerce_raw/
# ============================================================

import pandas as pd
import numpy as np

np.random.seed(42)

# ------------------------------------------------------------
# Target catalog/schema/volume
# ------------------------------------------------------------

CATALOG = "hackathon"
BRONZE_SCHEMA = "ecommerce_hypermarketing_dev_bronze"
VOLUME = "ecommerce_raw"

RAW_VOLUME = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/{VOLUME}"

print("Target raw volume:", RAW_VOLUME)

# ------------------------------------------------------------
# Optional: create schema and volume if you have permissions
# ------------------------------------------------------------

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{BRONZE_SCHEMA}`")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{CATALOG}`.`{BRONZE_SCHEMA}`.`{VOLUME}`")

# ------------------------------------------------------------
# Shared seed data
# ------------------------------------------------------------

current_hour = pd.Timestamp.now().floor("H")
hours = pd.date_range(end=current_hour, periods=48, freq="H")

products = pd.DataFrame({
    "SKU": [
        "EL-4821", "FA-1108", "BE-2044", "HM-3175",
        "SP-1451", "GR-5632", "EL-5540", "FA-2213"
    ],
    "Product_Name": [
        "Wireless Earbuds",
        "Summer Linen Shirt",
        "Vitamin C Serum",
        "Air Fryer 4L",
        "Yoga Mat Pro",
        "Healthy Snack Box",
        "Smartwatch Lite",
        "Running Shorts"
    ],
    "Category": [
        "Electronics",
        "Fashion",
        "Beauty",
        "Home",
        "Sports",
        "Grocery",
        "Electronics",
        "Fashion"
    ],
    "Brand": [
        "SoundPeak",
        "UrbanWeave",
        "GlowLab",
        "HomeEase",
        "FlexFit",
        "SnackWell",
        "PulseTime",
        "SprintWear"
    ],
    "Unit_Price": [
        79.00, 34.00, 24.00, 129.00,
        32.00, 19.00, 119.00, 28.00
    ],
    "Margin_Pct": [
        38.00, 44.00, 57.00, 31.00,
        46.00, 29.00, 34.00, 42.00
    ],
    "Base_Stock": [
        1800, 2400, 2600, 900,
        1700, 3200, 1200, 2100
    ],
    "Initial_Stock_Cover_Hrs": [
        9.00, 14.00, 18.00, 22.00,
        19.00, 31.00, 16.00, 20.00
    ],
    "Base_Units_Sold_Per_Hour": [
        125, 95, 75, 38,
        48, 32, 66, 44
    ]
})

product_settings = {
    "EL-4821": {"base_sessions": 1500, "ctr": 4.8, "cvr": 5.1, "demand_spike": 38, "roas_target": 6.2},
    "FA-1108": {"base_sessions": 1300, "ctr": 3.9, "cvr": 4.2, "demand_spike": 26, "roas_target": 4.7},
    "BE-2044": {"base_sessions": 1200, "ctr": 4.2, "cvr": 4.8, "demand_spike": 24, "roas_target": 5.4},
    "HM-3175": {"base_sessions": 700,  "ctr": 2.9, "cvr": 2.7, "demand_spike": 20, "roas_target": 3.1},
    "SP-1451": {"base_sessions": 900,  "ctr": 3.4, "cvr": 3.3, "demand_spike": 18, "roas_target": 4.0},
    "GR-5632": {"base_sessions": 650,  "ctr": 2.1, "cvr": 2.3, "demand_spike": 7,  "roas_target": 2.2},
    "EL-5540": {"base_sessions": 1000, "ctr": 4.5, "cvr": 4.7, "demand_spike": 28, "roas_target": 5.7},
    "FA-2213": {"base_sessions": 850,  "ctr": 3.0, "cvr": 3.5, "demand_spike": 16, "roas_target": 3.8},
}

intraday_pattern = np.array([
    0.72, 0.68, 0.64, 0.61, 0.63, 0.71,
    0.84, 0.96, 1.04, 1.10, 1.15, 1.20,
    1.24, 1.27, 1.31, 1.36, 1.43, 1.49,
    1.55, 1.49, 1.38, 1.24, 1.08, 0.92
])

# ------------------------------------------------------------
# 1. product_master.csv
# ------------------------------------------------------------

product_master_df = products[[
    "SKU",
    "Product_Name",
    "Category",
    "Brand",
    "Unit_Price",
    "Margin_Pct",
    "Base_Stock",
    "Initial_Stock_Cover_Hrs"
]].copy()

# ------------------------------------------------------------
# 2. marketing_performance_hourly.csv
# ------------------------------------------------------------

marketing_rows = []

for ts in hours:
    hour_factor = intraday_pattern[ts.hour]

    total_range_seconds = max(
        1,
        (hours.max() - hours.min()).total_seconds()
    )

    trend_factor = 1.0 + (
        (ts - hours.min()).total_seconds() / total_range_seconds
    ) * 0.08

    base_demand_index = max(
        52,
        100 * hour_factor * trend_factor + np.random.normal(0, 3.5)
    )

    for product in products.itertuples(index=False):
        settings = product_settings[product.SKU]

        category_effect = {
            "Electronics": 1.10,
            "Beauty": 1.05,
            "Fashion": 1.00,
            "Home": 0.86,
            "Sports": 0.92,
            "Grocery": 0.76
        }[product.Category]

        sessions = max(
            50,
            int(
                settings["base_sessions"]
                * (base_demand_index / 100)
                * category_effect
                * np.random.uniform(0.92, 1.08)
            )
        )

        ctr_pct = max(1.0, settings["ctr"] + np.random.normal(0, 0.18))
        cvr_pct = max(1.0, settings["cvr"] + np.random.normal(0, 0.16))

        clicks = int(round(sessions * ctr_pct / 100))
        orders = max(1, int(round(sessions * cvr_pct / 100)))
        units_sold = max(1, int(round(orders * np.random.uniform(1.15, 1.35))))

        gross_revenue = round(
            units_sold * product.Unit_Price * np.random.uniform(0.96, 1.04),
            2
        )

        roas_target = max(
            1.5,
            settings["roas_target"] + np.random.normal(0, 0.25)
        )

        ad_spend = round(gross_revenue / roas_target, 2)
        roas = round(gross_revenue / ad_spend, 2) if ad_spend else 0

        demand_spike_pct = max(
            0,
            settings["demand_spike"] + np.random.normal(0, 2.2)
        )

        discount_pct = round(
            np.clip(
                np.random.normal(
                    5 if product.Category in ["Electronics", "Fashion"] else 3.5,
                    1.8
                ),
                0,
                12
            ),
            1
        )

        avg_dwell_time_sec = int(
            round(
                np.clip(
                    np.random.normal(60, 10),
                    25,
                    120
                )
            )
        )

        push_score = round(
            0.27 * demand_spike_pct +
            0.18 * roas * 10 +
            0.18 * cvr_pct * 10 +
            0.17 * product.Margin_Pct +
            0.10 * min(100, ctr_pct * 12),
            1
        )

        marketing_rows.append({
            "Timestamp": ts,
            "Hour_Key": ts.strftime("%Y%m%d%H"),
            "SKU": product.SKU,
            "Product_Name": product.Product_Name,
            "Category": product.Category,
            "Sessions": sessions,
            "Clicks": clicks,
            "CTR_Pct": round(ctr_pct, 2),
            "Orders": orders,
            "Units_Sold": units_sold,
            "CVR_Pct": round(cvr_pct, 2),
            "Gross_Revenue": gross_revenue,
            "Ad_Spend": ad_spend,
            "ROAS": roas,
            "Demand_Index": round(base_demand_index, 1),
            "Demand_Spike_Pct": round(demand_spike_pct, 1),
            "Discount_Pct": discount_pct,
            "Avg_Dwell_Time_Sec": avg_dwell_time_sec,
            "Push_Score": push_score
        })

marketing_df = pd.DataFrame(marketing_rows)

# ------------------------------------------------------------
# 3. stock_movement_hourly.csv
# ------------------------------------------------------------

stock_rows = []

current_stock = {
    row.SKU: int(row.Base_Stock)
    for row in products.itertuples(index=False)
}

restock_plan = {
    "EL-4821": {9: 300, 38: 450},
    "FA-1108": {12: 400},
    "BE-2044": {20: 500},
    "HM-3175": {16: 200},
    "SP-1451": {36: 350},
    "GR-5632": {40: 600},
    "EL-5540": {18: 280},
    "FA-2213": {24: 260}
}

for hour_index, ts in enumerate(hours):
    hour_factor = intraday_pattern[ts.hour]

    total_range_seconds = max(
        1,
        (hours.max() - hours.min()).total_seconds()
    )

    trend_factor = 1.0 + (
        (ts - hours.min()).total_seconds() / total_range_seconds
    ) * 0.08

    for product in products.itertuples(index=False):
        sku = product.SKU
        opening_stock = int(current_stock[sku])

        category_effect = {
            "Electronics": 1.18,
            "Beauty": 1.08,
            "Fashion": 1.00,
            "Home": 0.78,
            "Sports": 0.86,
            "Grocery": 0.68
        }[product.Category]

        units_sold = int(
            max(
                1,
                round(
                    product.Base_Units_Sold_Per_Hour
                    * hour_factor
                    * trend_factor
                    * category_effect
                    * np.random.uniform(0.88, 1.12)
                )
            )
        )

        units_sold = min(units_sold, opening_stock)
        restocked_units = restock_plan.get(sku, {}).get(hour_index, 0)

        closing_stock = max(
            0,
            opening_stock - units_sold + restocked_units
        )

        current_stock[sku] = closing_stock

        stock_cover_hrs = round(
            closing_stock / max(units_sold, 1),
            1
        )

        if stock_cover_hrs < 12:
            stock_risk_flag = "High"
        elif stock_cover_hrs < 18:
            stock_risk_flag = "Medium"
        else:
            stock_risk_flag = "Low"

        stock_rows.append({
            "Timestamp": ts,
            "Hour_Key": ts.strftime("%Y%m%d%H"),
            "SKU": sku,
            "Product_Name": product.Product_Name,
            "Category": product.Category,
            "Opening_Stock": opening_stock,
            "Units_Sold": units_sold,
            "Restocked_Units": restocked_units,
            "Closing_Stock": closing_stock,
            "Stock_Cover_Hrs": stock_cover_hrs,
            "Stock_Risk_Flag": stock_risk_flag
        })

stock_df = pd.DataFrame(stock_rows)

# ------------------------------------------------------------
# 4. order_transactions_sample.csv
# ------------------------------------------------------------

traffic_channels = ["Search", "Social", "App Push", "Marketplace Ads", "Display"]
traffic_channel_prob = [0.30, 0.24, 0.18, 0.18, 0.10]

cities = ["Bangalore", "Hyderabad", "Chennai", "Mumbai", "Pune"]
city_prob = [0.35, 0.22, 0.16, 0.16, 0.11]

customer_types = ["New", "Returning"]
customer_type_prob = [0.38, 0.62]

payment_modes = ["UPI", "Card", "COD", "Wallet"]
payment_mode_prob = [0.44, 0.31, 0.13, 0.12]

product_order_intensity = {
    "EL-4821": 1.35,
    "FA-1108": 1.10,
    "BE-2044": 1.15,
    "HM-3175": 0.65,
    "SP-1451": 0.78,
    "GR-5632": 0.55,
    "EL-5540": 1.05,
    "FA-2213": 0.75
}

transaction_rows = []
order_counter = 100000

for ts in hours:
    hour_factor = intraday_pattern[ts.hour]

    total_range_seconds = max(
        1,
        (hours.max() - hours.min()).total_seconds()
    )

    trend_factor = 1.0 + (
        (ts - hours.min()).total_seconds() / total_range_seconds
    ) * 0.08

    for product in products.itertuples(index=False):
        sku = product.SKU

        base_txn_count = int(
            max(
                2,
                round(
                    8
                    * product_order_intensity[sku]
                    * hour_factor
                    * trend_factor
                    * np.random.uniform(0.80, 1.25)
                )
            )
        )

        for _ in range(base_txn_count):
            order_counter += 1

            order_timestamp = ts + pd.Timedelta(
                minutes=int(np.random.uniform(0, 60)),
                seconds=int(np.random.uniform(0, 60))
            )

            quantity = int(
                max(
                    1,
                    np.random.choice(
                        [1, 1, 1, 2, 2, 3],
                        p=[0.45, 0.22, 0.13, 0.12, 0.06, 0.02]
                    )
                )
            )

            discount_pct = round(
                float(np.clip(np.random.normal(4.5, 1.7), 0, 12)),
                1
            )

            gross_value = product.Unit_Price * quantity
            order_value = round(
                gross_value
                * (1 - discount_pct / 100)
                * np.random.uniform(0.98, 1.02),
                2
            )

            transaction_rows.append({
                "Order_ID": f"ORD{order_counter}",
                "Order_Timestamp": order_timestamp,
                "SKU": product.SKU,
                "Product_Name": product.Product_Name,
                "Category": product.Category,
                "Quantity": quantity,
                "Order_Value": order_value,
                "Discount_Pct": discount_pct,
                "Traffic_Channel": np.random.choice(traffic_channels, p=traffic_channel_prob),
                "City": np.random.choice(cities, p=city_prob),
                "Customer_Type": np.random.choice(customer_types, p=customer_type_prob),
                "Payment_Mode": np.random.choice(payment_modes, p=payment_mode_prob)
            })

order_transactions_df = pd.DataFrame(transaction_rows)
order_transactions_df = order_transactions_df.sort_values(
    ["Order_Timestamp", "SKU", "Order_ID"]
).reset_index(drop=True)

# ------------------------------------------------------------
# 5. orders_hourly_summary.csv
# ------------------------------------------------------------

orders_hourly_df = (
    marketing_df
    .groupby(["Timestamp", "Hour_Key"], as_index=False)
    .agg(
        Total_Orders=("Orders", "sum"),
        Total_Units=("Units_Sold", "sum"),
        Total_Sessions=("Sessions", "sum"),
        Revenue=("Gross_Revenue", "sum"),
        Ad_Spend=("Ad_Spend", "sum")
    )
)

orders_hourly_df["Conversion_Rate_Pct"] = (
    orders_hourly_df["Total_Orders"]
    / orders_hourly_df["Total_Sessions"]
    * 100
).round(2)

orders_hourly_df["AOV"] = (
    orders_hourly_df["Revenue"]
    / orders_hourly_df["Total_Orders"]
).round(2)

orders_hourly_df["Revenue"] = orders_hourly_df["Revenue"].round(2)
orders_hourly_df["Ad_Spend"] = orders_hourly_df["Ad_Spend"].round(2)

# ------------------------------------------------------------
# 6. category_performance_hourly.csv
# ------------------------------------------------------------

category_perf_df = (
    marketing_df
    .groupby(["Timestamp", "Hour_Key", "Category"], as_index=False)
    .agg(
        Sessions=("Sessions", "sum"),
        Orders=("Orders", "sum"),
        Revenue=("Gross_Revenue", "sum"),
        Ad_Spend=("Ad_Spend", "sum"),
        Avg_Demand_Spike_Pct=("Demand_Spike_Pct", "mean"),
        Avg_Push_Score=("Push_Score", "mean")
    )
)

category_perf_df["CVR_Pct"] = (
    category_perf_df["Orders"]
    / category_perf_df["Sessions"]
    * 100
).round(2)

category_perf_df["ROAS"] = (
    category_perf_df["Revenue"]
    / category_perf_df["Ad_Spend"]
).round(2)

category_perf_df["Revenue"] = category_perf_df["Revenue"].round(2)
category_perf_df["Ad_Spend"] = category_perf_df["Ad_Spend"].round(2)
category_perf_df["Avg_Demand_Spike_Pct"] = category_perf_df["Avg_Demand_Spike_Pct"].round(1)
category_perf_df["Avg_Push_Score"] = category_perf_df["Avg_Push_Score"].round(1)

# ------------------------------------------------------------
# 7. channel_performance_hourly.csv
# ------------------------------------------------------------

channel_rows = []

channel_mix = {
    "Search": 0.30,
    "Social": 0.24,
    "App Push": 0.16,
    "Marketplace Ads": 0.18,
    "Display": 0.12
}

channel_cvr_boost = {
    "Search": 1.12,
    "Social": 0.98,
    "App Push": 1.18,
    "Marketplace Ads": 0.94,
    "Display": 0.82
}

channel_roas_boost = {
    "Search": 1.08,
    "Social": 0.96,
    "App Push": 1.20,
    "Marketplace Ads": 0.88,
    "Display": 0.74
}

for row in orders_hourly_df.itertuples(index=False):
    for channel_name in channel_mix.keys():
        sessions = int(
            round(
                row.Total_Sessions
                * channel_mix[channel_name]
                * np.random.uniform(0.92, 1.08)
            )
        )

        orders = int(
            round(
                row.Total_Orders
                * channel_mix[channel_name]
                * channel_cvr_boost[channel_name]
                * np.random.uniform(0.90, 1.10)
            )
        )

        revenue = round(
            row.Revenue
            * channel_mix[channel_name]
            * channel_roas_boost[channel_name]
            * np.random.uniform(0.92, 1.08),
            2
        )

        ad_spend = round(
            max(
                50,
                revenue / np.random.uniform(2.5, 6.5)
            ),
            2
        )

        cvr_pct = round(
            orders / max(sessions, 1) * 100,
            2
        )

        roas = round(
            revenue / max(ad_spend, 1),
            2
        )

        channel_rows.append({
            "Timestamp": row.Timestamp,
            "Hour_Key": row.Hour_Key,
            "Channel": channel_name,
            "Sessions": sessions,
            "Orders": orders,
            "Revenue": revenue,
            "Ad_Spend": ad_spend,
            "CVR_Pct": cvr_pct,
            "ROAS": roas
        })

channel_perf_df = pd.DataFrame(channel_rows)

# ------------------------------------------------------------
# 8. restock_schedule.csv
# ------------------------------------------------------------

restock_rows = [
    ["EL-4821", "Wireless Earbuds", current_hour + pd.Timedelta(hours=2), 450, "Planned"],
    ["FA-1108", "Summer Linen Shirt", current_hour + pd.Timedelta(hours=3), 400, "Planned"],
    ["BE-2044", "Vitamin C Serum", current_hour + pd.Timedelta(hours=4), 500, "Planned"],
    ["HM-3175", "Air Fryer 4L", current_hour + pd.Timedelta(hours=5), 200, "Planned"],
    ["SP-1451", "Yoga Mat Pro", current_hour + pd.Timedelta(hours=6), 350, "Planned"],
    ["GR-5632", "Healthy Snack Box", current_hour + pd.Timedelta(hours=7), 600, "Planned"],
    ["EL-5540", "Smartwatch Lite", current_hour + pd.Timedelta(hours=8), 280, "Planned"],
    ["FA-2213", "Running Shorts", current_hour + pd.Timedelta(hours=9), 260, "Planned"],
]

restock_schedule_df = pd.DataFrame(
    restock_rows,
    columns=[
        "SKU",
        "Product_Name",
        "Scheduled_Restock_Timestamp",
        "Restock_Units",
        "Status"
    ]
)

# ------------------------------------------------------------
# Write all files directly to UC Volume using dbutils.fs.put
# ------------------------------------------------------------

outputs = {
    "product_master.csv": product_master_df,
    "marketing_performance_hourly.csv": marketing_df,
    "stock_movement_hourly.csv": stock_df,
    "order_transactions_sample.csv": order_transactions_df,
    "orders_hourly_summary.csv": orders_hourly_df,
    "category_performance_hourly.csv": category_perf_df,
    "channel_performance_hourly.csv": channel_perf_df,
    "restock_schedule.csv": restock_schedule_df,
}

for file_name, df in outputs.items():
    target_path = f"{RAW_VOLUME}/{file_name}"
    csv_data = df.to_csv(index=False)

    dbutils.fs.put(
        target_path,
        csv_data,
        overwrite=True
    )

    print(f"Created: {target_path}")
    print(f"Rows: {len(df)}")
    print("-" * 80)

print("All CSV files created successfully.")

Target raw volume: /Volumes/hackathon/ecommerce_hypermarketing_dev_bronze/ecommerce_raw


/home/spark-75af134b-bc6f-47f2-bacf-d8/.ipykernel/2076/command-8778560173977081-1266331618:35: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  current_hour = pd.Timestamp.now().floor("H")
/home/spark-75af134b-bc6f-47f2-bacf-d8/.ipykernel/2076/command-8778560173977081-1266331618:36: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hours = pd.date_range(end=current_hour, periods=48, freq="H")


Wrote 584 bytes.
Created: /Volumes/hackathon/ecommerce_hypermarketing_dev_bronze/ecommerce_raw/product_master.csv
Rows: 8
--------------------------------------------------------------------------------
Wrote 49777 bytes.
Created: /Volumes/hackathon/ecommerce_hypermarketing_dev_bronze/ecommerce_raw/marketing_performance_hourly.csv
Rows: 384
--------------------------------------------------------------------------------
Wrote 32755 bytes.
Created: /Volumes/hackathon/ecommerce_hypermarketing_dev_bronze/ecommerce_raw/stock_movement_hourly.csv
Rows: 384
--------------------------------------------------------------------------------
Wrote 342369 bytes.
Created: /Volumes/hackathon/ecommerce_hypermarketing_dev_bronze/ecommerce_raw/order_transactions_sample.csv
Rows: 3255
--------------------------------------------------------------------------------
Wrote 3564 bytes.
Created: /Volumes/hackathon/ecommerce_hypermarketing_dev_bronze/ecommerce_raw/orders_hourly_summary.csv
Rows: 48
-----------

In [0]:
# COMMAND ----------

# ============================================================
# 5. Bronze ingestion - Unity Catalog compatible
# ============================================================

from pyspark.sql import functions as F

for source_name, meta in TABLE_REGISTRY.items():
    bronze_table = qname(BRONZE, meta["bronze"])
    file_path = f"{RAW_VOLUME}/{meta['source_file']}"

    print("=" * 80)
    print(f"Loading source: {source_name}")
    print(f"File path: {file_path}")
    print(f"Bronze table: {bronze_table}")

    try:
        # Read CSV from Unity Catalog Volume
        df_raw = (
            spark.read
            .format("csv")
            .option("header", True)
            .option("inferSchema", True)
            .option("multiLine", True)
            .option("escape", '"')
            .load(file_path)
        )

        # Add file metadata using Unity Catalog supported _metadata.file_path
        # Do NOT use F.input_file_name() in Unity Catalog.
        try:
            df_raw = df_raw.withColumn("_file_path", F.col("_metadata.file_path"))
        except Exception:
            # Fallback for direct file reads if _metadata is unavailable
            df_raw = df_raw.withColumn("_file_path", F.lit(file_path))

        # Normalize columns after metadata column is added
        df = normalize_columns(df_raw)

        # Add ingestion metadata
        df = (
            df.withColumn("_source_name", F.lit(source_name))
              .withColumn("_ingested_at", F.current_timestamp())
        )

        # Write to Bronze Delta table
        (
            df.write
            .format("delta")
            .mode("append")
            .option("mergeSchema", True)
            .saveAsTable(bronze_table)
        )

        row_count = df.count()

        try:
            audit("bronze_ingestion", bronze_table, "SUCCESS", row_count, None)
        except Exception as audit_exc:
            print(f"WARNING: Audit failed for {bronze_table}: {audit_exc}")

        print(f"Bronze loaded successfully: {bronze_table}")
        print(f"Rows loaded: {row_count}")

    except Exception as exc:
        try:
            audit("bronze_ingestion", bronze_table, "FAILED", None, str(exc))
        except Exception as audit_exc:
            print(f"WARNING: Audit failed for failed bronze load: {audit_exc}")

        print(f"FAILED loading Bronze table: {bronze_table}")
        print("Error:", str(exc))
        raise

Loading source: product_master
File path: /Volumes/hackathon/ecommerce_hypermarketing_dev_bronze/ecommerce_raw/product_master.csv
Bronze table: `hackathon`.`ecommerce_hypermarketing_dev_bronze`.`raw_product_master`


/home/spark-75af134b-bc6f-47f2-bacf-d8/.ipykernel/2076/command-8778560173977054-425456714:34: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  rows = [(RUN_ID, task_name, table_name, status, row_count, message, datetime.utcnow(), datetime.utcnow())]


Bronze loaded successfully: `hackathon`.`ecommerce_hypermarketing_dev_bronze`.`raw_product_master`
Rows loaded: 8
Loading source: marketing_performance_hourly
File path: /Volumes/hackathon/ecommerce_hypermarketing_dev_bronze/ecommerce_raw/marketing_performance_hourly.csv
Bronze table: `hackathon`.`ecommerce_hypermarketing_dev_bronze`.`raw_marketing_performance_hourly`
Bronze loaded successfully: `hackathon`.`ecommerce_hypermarketing_dev_bronze`.`raw_marketing_performance_hourly`
Rows loaded: 384
Loading source: stock_movement_hourly
File path: /Volumes/hackathon/ecommerce_hypermarketing_dev_bronze/ecommerce_raw/stock_movement_hourly.csv
Bronze table: `hackathon`.`ecommerce_hypermarketing_dev_bronze`.`raw_stock_movement_hourly`
Bronze loaded successfully: `hackathon`.`ecommerce_hypermarketing_dev_bronze`.`raw_stock_movement_hourly`
Rows loaded: 384
Loading source: order_transactions
File path: /Volumes/hackathon/ecommerce_hypermarketing_dev_bronze/ecommerce_raw/order_transactions_sample

In [0]:


# COMMAND ----------

# ============================================================
# 6. Silver build: type casting, dedupe, DQ, upsert
# ============================================================

for source_name, meta in TABLE_REGISTRY.items():
    bronze_table = qname(BRONZE, meta["bronze"])
    silver_table = qname(SILVER, meta["silver"])
    keys = meta["keys"]
    cols = meta["columns"]

    try:
        df = normalize_columns(spark.table(bronze_table))
        for col_name, dtype in cols.items():
            if col_name in df.columns:
                df = df.withColumn(col_name, F.col(col_name).cast(dtype))
            else:
                df = df.withColumn(col_name, F.lit(None).cast(dtype))

        selected = list(cols.keys()) + [c for c in ["_source_name", "_ingested_at", "_input_file_name"] if c in df.columns]
        df = df.select(*selected).dropDuplicates(keys).withColumn("_silver_processed_at", F.current_timestamp())

        for key in keys:
            failed = df.filter(F.col(key).isNull()).count()
            dq_result("silver", meta["silver"], f"not_null:{key}", "PASS" if failed == 0 else "FAIL", failed)

        dupes = df.groupBy(*keys).count().filter(F.col("count") > 1).count()
        dq_result("silver", meta["silver"], "unique_key:" + ",".join(keys), "PASS" if dupes == 0 else "FAIL", dupes)

        row_count = merge_upsert(df, silver_table, keys)
        audit("silver_build", silver_table, "SUCCESS", row_count, None)
        print("Silver built:", silver_table)
    except Exception as exc:
        audit("silver_build", silver_table, "FAILED", None, str(exc))
        raise


/home/spark-75af134b-bc6f-47f2-bacf-d8/.ipykernel/2076/command-8778560173977054-425456714:40: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  rows = [(RUN_ID, layer, table_name, rule_name, rule_status, int(failed_count), datetime.utcnow())]
/home/spark-75af134b-bc6f-47f2-bacf-d8/.ipykernel/2076/command-8778560173977054-425456714:34: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  rows = [(RUN_ID, task_name, table_name, status, row_count, message, datetime.utcnow(), datetime.utcnow())]


Silver built: `hackathon`.`ecommerce_hypermarketing_dev_silver`.`dim_product`
Silver built: `hackathon`.`ecommerce_hypermarketing_dev_silver`.`fact_marketing_performance_hourly`
Silver built: `hackathon`.`ecommerce_hypermarketing_dev_silver`.`fact_stock_movement_hourly`
Silver built: `hackathon`.`ecommerce_hypermarketing_dev_silver`.`fact_order_transactions`
Silver built: `hackathon`.`ecommerce_hypermarketing_dev_silver`.`fact_orders_hourly_summary`
Silver built: `hackathon`.`ecommerce_hypermarketing_dev_silver`.`fact_category_performance_hourly`
Silver built: `hackathon`.`ecommerce_hypermarketing_dev_silver`.`fact_channel_performance_hourly`
Silver built: `hackathon`.`ecommerce_hypermarketing_dev_silver`.`fact_restock_schedule`


In [0]:
# COMMAND ----------

# ============================================================
# 7. Gold build: main KPI, category lift, push-now,
#    action center, budget shift, low-stock throttle view
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql import Window
from decimal import Decimal

# ------------------------------------------------------------
# Safe threshold defaults
# ------------------------------------------------------------

STOCK_LOW_HRS = float(STOCK_LOW_HRS) if "STOCK_LOW_HRS" in globals() else 12.0
MIN_PUSH_ROAS = float(MIN_PUSH_ROAS) if "MIN_PUSH_ROAS" in globals() else 5.0
MIN_PUSH_CVR = float(MIN_PUSH_CVR) if "MIN_PUSH_CVR" in globals() else 4.5

print("Using thresholds:")
print("STOCK_LOW_HRS:", STOCK_LOW_HRS)
print("MIN_PUSH_ROAS:", MIN_PUSH_ROAS)
print("MIN_PUSH_CVR:", MIN_PUSH_CVR)

# ------------------------------------------------------------
# Validate catalog/schema variables
# ------------------------------------------------------------

print("CATALOG:", CATALOG)
print("SILVER:", SILVER)
print("GOLD:", GOLD)
print("OPS:", OPS)

# ------------------------------------------------------------
# Read Silver tables
# ------------------------------------------------------------

m = spark.table(qname(SILVER, "fact_marketing_performance_hourly"))
s = spark.table(qname(SILVER, "fact_stock_movement_hourly"))
p = spark.table(qname(SILVER, "dim_product"))
cat = spark.table(qname(SILVER, "fact_category_performance_hourly"))
ch = spark.table(qname(SILVER, "fact_channel_performance_hourly"))

# ------------------------------------------------------------
# Cast Decimal / numeric columns to Spark double/long
# This prevents Python Decimal * float errors after collect()
# and keeps Spark calculations consistent.
# ------------------------------------------------------------

m = (
    m.withColumn("gross_revenue", F.col("gross_revenue").cast("double"))
     .withColumn("ad_spend", F.col("ad_spend").cast("double"))
     .withColumn("roas", F.col("roas").cast("double"))
     .withColumn("demand_index", F.col("demand_index").cast("double"))
     .withColumn("demand_spike_pct", F.col("demand_spike_pct").cast("double"))
     .withColumn("cvr_pct", F.col("cvr_pct").cast("double"))
     .withColumn("push_score", F.col("push_score").cast("double"))
     .withColumn("orders", F.col("orders").cast("long"))
     .withColumn("units_sold", F.col("units_sold").cast("long"))
     .withColumn("sessions", F.col("sessions").cast("long"))
)

s = (
    s.withColumn("closing_stock", F.col("closing_stock").cast("long"))
     .withColumn("stock_cover_hrs", F.col("stock_cover_hrs").cast("double"))
)

p = (
    p.withColumn("margin_pct", F.col("margin_pct").cast("double"))
     .withColumn("unit_price", F.col("unit_price").cast("double"))
)

cat = (
    cat.withColumn("avg_demand_spike_pct", F.col("avg_demand_spike_pct").cast("double"))
       .withColumn("avg_push_score", F.col("avg_push_score").cast("double"))
       .withColumn("cvr_pct", F.col("cvr_pct").cast("double"))
       .withColumn("roas", F.col("roas").cast("double"))
)

ch = (
    ch.withColumn("revenue", F.col("revenue").cast("double"))
      .withColumn("ad_spend", F.col("ad_spend").cast("double"))
      .withColumn("cvr_pct", F.col("cvr_pct").cast("double"))
      .withColumn("roas", F.col("roas").cast("double"))
      .withColumn("sessions", F.col("sessions").cast("long"))
      .withColumn("orders", F.col("orders").cast("long"))
)

# ------------------------------------------------------------
# Latest timestamp selection
# ------------------------------------------------------------

latest_ts = m.agg(F.max("timestamp").alias("ts")).collect()[0]["ts"]

if latest_ts is None:
    raise ValueError("No data found in Silver table fact_marketing_performance_hourly")

perf_latest = m.filter(F.col("timestamp") == F.lit(latest_ts))
stock_latest = s.filter(F.col("timestamp") == F.lit(latest_ts))

print("Latest marketing timestamp:", latest_ts)

# ------------------------------------------------------------
# 7.1 Main KPI table
# ------------------------------------------------------------

kpi = (
    perf_latest.groupBy("timestamp", "hour_key")
    .agg(
        F.sum("gross_revenue").alias("revenue_last_hour"),
        F.sum("orders").alias("orders_last_hour"),
        F.sum("units_sold").alias("units_last_hour"),
        F.sum("sessions").alias("sessions_last_hour"),
        (
            F.sum("orders").cast("double")
            / F.when(F.sum("sessions") == 0, F.lit(None))
               .otherwise(F.sum("sessions"))
               .cast("double")
            * F.lit(100.0)
        ).alias("conversion_rate_pct"),
        (
            F.sum("gross_revenue")
            / F.when(F.sum("ad_spend") == 0, F.lit(None))
               .otherwise(F.sum("ad_spend"))
        ).alias("roas"),
        F.avg("demand_index").alias("demand_pulse_index")
    )
)

stock_total = (
    stock_latest
    .groupBy("timestamp")
    .agg(F.sum("closing_stock").alias("stock_on_hand_total"))
)

main_kpis = (
    kpi.join(stock_total, "timestamp", "left")
       .withColumn("created_at", F.current_timestamp())
)

(
    main_kpis.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(qname(GOLD, "main_dashboard_kpis"))
)

print("Gold table built:", qname(GOLD, "main_dashboard_kpis"))

# ------------------------------------------------------------
# 7.2 Category demand lift
# ------------------------------------------------------------

cat_latest_ts = cat.agg(F.max("timestamp").alias("ts")).collect()[0]["ts"]

if cat_latest_ts is None:
    raise ValueError("No data found in Silver table fact_category_performance_hourly")

category_lift = (
    cat.filter(F.col("timestamp") == F.lit(cat_latest_ts))
       .select(
           "timestamp",
           "category",
           F.col("avg_demand_spike_pct").alias("demand_lift_pct"),
           "avg_push_score",
           "cvr_pct",
           "roas"
       )
       .withColumn("created_at", F.current_timestamp())
)

(
    category_lift.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(qname(GOLD, "category_demand_lift_current"))
)

print("Gold table built:", qname(GOLD, "category_demand_lift_current"))

# ------------------------------------------------------------
# 7.3 Product push-now table
# ------------------------------------------------------------

stock_select = (
    stock_latest
    .select(
        "hour_key",
        "sku",
        "closing_stock",
        "stock_cover_hrs",
        "stock_risk_flag"
    )
)

product_select = (
    p.select(
        "sku",
        "margin_pct",
        "unit_price",
        "brand"
    )
)

push_base = (
    perf_latest
    .join(stock_select, ["hour_key", "sku"], "left")
    .join(product_select, ["sku"], "left")
)

ranking_window = Window.orderBy(
    F.desc("push_score"),
    F.desc("roas"),
    F.desc("cvr_pct")
)

push_now = (
    push_base
    .withColumn(
        "decision",
        F.when(
            F.col("stock_cover_hrs") < F.lit(float(STOCK_LOW_HRS)),
            F.lit("PROMOTE_CAREFULLY_LOW_STOCK")
        )
        .when(
            (F.col("roas") >= F.lit(float(MIN_PUSH_ROAS))) &
            (F.col("cvr_pct") >= F.lit(float(MIN_PUSH_CVR))),
            F.lit("PUSH_NOW_PAID_AND_OWNED")
        )
        .when(
            F.col("margin_pct") >= F.lit(45.0),
            F.lit("PUSH_HIGH_INTENT_PROFIT")
        )
        .otherwise(F.lit("SECONDARY_PUSH"))
    )
    .withColumn(
        "action_type",
        F.when(
            F.col("decision") == F.lit("PUSH_NOW_PAID_AND_OWNED"),
            F.lit("Push Now")
        )
        .when(
            F.col("decision") == F.lit("PROMOTE_CAREFULLY_LOW_STOCK"),
            F.lit("Guarded Push")
        )
        .otherwise(F.lit("Secondary Push"))
    )
    .withColumn(
        "primary_channel_sequence",
        F.when(
            F.col("action_type") == F.lit("Push Now"),
            F.lit("Homepage Tile → App Push → Search Boost → Retargeting")
        )
        .when(
            F.col("action_type") == F.lit("Guarded Push"),
            F.lit("Retargeting Only → Frequency Cap → Await Restock")
        )
        .otherwise(F.lit("Category Placement → Search Boost → Retargeting"))
    )
    .withColumn(
        "suggested_budget_shift_pct",
        F.when(
            F.col("action_type") == F.lit("Push Now"),
            F.lit(12.0)
        )
        .when(
            F.col("action_type") == F.lit("Secondary Push"),
            F.lit(5.0)
        )
        .otherwise(F.lit(0.0))
    )
    .withColumn("priority_rank", F.row_number().over(ranking_window))
    .withColumn("created_at", F.current_timestamp())
)

(
    push_now.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(qname(GOLD, "product_pushnow_current"))
)

print("Gold table built:", qname(GOLD, "product_pushnow_current"))

# ------------------------------------------------------------
# 7.4 Recommended actions
# ------------------------------------------------------------

recommended_actions = (
    push_now
    .filter(F.col("priority_rank") <= F.lit(5))
    .withColumn(
        "reason",
        F.concat(
            F.lit("Demand spike "),
            F.col("demand_spike_pct").cast("string"),
            F.lit("%, CVR "),
            F.col("cvr_pct").cast("string"),
            F.lit("%, ROAS "),
            F.col("roas").cast("string"),
            F.lit("x, stock cover "),
            F.col("stock_cover_hrs").cast("string"),
            F.lit(" hrs")
        )
    )
    .withColumn(
        "keep_promoting_until_hour",
        F.expr("timestamp + INTERVAL 3 HOURS")
    )
    .select(
        "priority_rank",
        "sku",
        "product_name",
        "category",
        "action_type",
        "primary_channel_sequence",
        "reason",
        "suggested_budget_shift_pct",
        "keep_promoting_until_hour"
    )
    .withColumn("created_at", F.current_timestamp())
)

(
    recommended_actions.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(qname(GOLD, "recommended_actions_current_hour"))
)

print("Gold table built:", qname(GOLD, "recommended_actions_current_hour"))

# ------------------------------------------------------------
# 7.5 Channel budget shift
# ------------------------------------------------------------

latest_ch_ts = ch.agg(F.max("timestamp").alias("ts")).collect()[0]["ts"]

if latest_ch_ts is None:
    raise ValueError("No data found in Silver table fact_channel_performance_hourly")

ch_latest = ch.filter(F.col("timestamp") == F.lit(latest_ch_ts))

avg_roas_raw = (
    ch_latest
    .agg(F.avg("roas").alias("avg_roas"))
    .collect()[0]["avg_roas"]
)

# avg_roas_raw can be decimal.Decimal or Python float.
# Convert to float before arithmetic.
avg_roas = float(avg_roas_raw) if avg_roas_raw is not None else 0.0
avg_roas_threshold_high = avg_roas * 1.15
avg_roas_threshold_mid = avg_roas

print("Latest channel timestamp:", latest_ch_ts)
print("Average channel ROAS:", avg_roas)
print("High ROAS threshold:", avg_roas_threshold_high)

budget_shift = (
    ch_latest
    .withColumn(
        "suggested_budget_shift_pct",
        F.when(
            F.col("roas") >= F.lit(avg_roas_threshold_high),
            F.lit(12.0)
        )
        .when(
            F.col("roas") >= F.lit(avg_roas_threshold_mid),
            F.lit(5.0)
        )
        .otherwise(F.lit(-8.0))
    )
    .withColumn(
        "budget_action",
        F.when(
            F.col("suggested_budget_shift_pct") > F.lit(0.0),
            F.lit("Increase")
        )
        .otherwise(F.lit("Reduce"))
    )
    .withColumn("created_at", F.current_timestamp())
)

(
    budget_shift.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(qname(GOLD, "channel_budget_shift_current"))
)

print("Gold table built:", qname(GOLD, "channel_budget_shift_current"))

# ------------------------------------------------------------
# 7.6 Low stock throttle persistent view
# IMPORTANT:
# Persistent Unity Catalog views cannot reference temporary views.
# Therefore this persistent view references the persistent Gold table.
# ------------------------------------------------------------

low_stock_view_name = qname(GOLD, "vw_low_stock_throttle_list")
push_now_table_name = qname(GOLD, "product_pushnow_current")

spark.sql(f"""
CREATE OR REPLACE VIEW {low_stock_view_name}
AS
SELECT
    sku,
    product_name,
    category,
    closing_stock,
    stock_cover_hrs,
    stock_risk_flag,
    action_type
FROM {push_now_table_name}
WHERE stock_cover_hrs < {float(STOCK_LOW_HRS)}
""")

print("Gold view built:", low_stock_view_name)

# ------------------------------------------------------------
# 7.7 Audit
# ------------------------------------------------------------

try:
    audit("gold_build", GOLD, "SUCCESS", push_now.count(), None)
except Exception as audit_exc:
    print("WARNING: Audit failed:", audit_exc)

print("Gold tables built successfully.")

Using thresholds:
STOCK_LOW_HRS: 12.0
MIN_PUSH_ROAS: 5.0
MIN_PUSH_CVR: 4.5
CATALOG: hackathon
SILVER: hackathon.ecommerce_hypermarketing_dev_silver
GOLD: hackathon.ecommerce_hypermarketing_dev_gold
OPS: hackathon.ecommerce_hypermarketing_dev_ops
Latest marketing timestamp: 2026-06-23 16:00:00
Gold table built: `hackathon`.`ecommerce_hypermarketing_dev_gold`.`main_dashboard_kpis`
Gold table built: `hackathon`.`ecommerce_hypermarketing_dev_gold`.`category_demand_lift_current`


/databricks/spark/python/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Gold table built: `hackathon`.`ecommerce_hypermarketing_dev_gold`.`product_pushnow_current`
Gold table built: `hackathon`.`ecommerce_hypermarketing_dev_gold`.`recommended_actions_current_hour`
Latest channel timestamp: 2026-06-23 16:00:00
Average channel ROAS: 4.926
High ROAS threshold: 5.664899999999999
Gold table built: `hackathon`.`ecommerce_hypermarketing_dev_gold`.`channel_budget_shift_current`
Gold view built: `hackathon`.`ecommerce_hypermarketing_dev_gold`.`vw_low_stock_throttle_list`


/home/spark-75af134b-bc6f-47f2-bacf-d8/.ipykernel/2076/command-8778560173977054-425456714:34: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  rows = [(RUN_ID, task_name, table_name, status, row_count, message, datetime.utcnow(), datetime.utcnow())]


Gold tables built successfully.


In [0]:


# COMMAND ----------

# ============================================================
# 8. Optimize Delta tables
# ============================================================

silver_tables = [meta["silver"] for meta in TABLE_REGISTRY.values()]
gold_tables = [
    "main_dashboard_kpis",
    "category_demand_lift_current",
    "product_pushnow_current",
    "recommended_actions_current_hour",
    "channel_budget_shift_current",
]

for table_name in silver_tables:
    try:
        spark.sql(f"OPTIMIZE {qname(SILVER, table_name)}")
        print("Optimized:", qname(SILVER, table_name))
    except Exception as exc:
        print("OPTIMIZE skipped/failed:", table_name, str(exc))

for table_name in gold_tables:
    try:
        spark.sql(f"OPTIMIZE {qname(GOLD, table_name)}")
        print("Optimized:", qname(GOLD, table_name))
    except Exception as exc:
        print("OPTIMIZE skipped/failed:", table_name, str(exc))


Optimized: `hackathon`.`ecommerce_hypermarketing_dev_silver`.`dim_product`
Optimized: `hackathon`.`ecommerce_hypermarketing_dev_silver`.`fact_marketing_performance_hourly`
Optimized: `hackathon`.`ecommerce_hypermarketing_dev_silver`.`fact_stock_movement_hourly`
Optimized: `hackathon`.`ecommerce_hypermarketing_dev_silver`.`fact_order_transactions`
Optimized: `hackathon`.`ecommerce_hypermarketing_dev_silver`.`fact_orders_hourly_summary`
Optimized: `hackathon`.`ecommerce_hypermarketing_dev_silver`.`fact_category_performance_hourly`
Optimized: `hackathon`.`ecommerce_hypermarketing_dev_silver`.`fact_channel_performance_hourly`
Optimized: `hackathon`.`ecommerce_hypermarketing_dev_silver`.`fact_restock_schedule`
Optimized: `hackathon`.`ecommerce_hypermarketing_dev_gold`.`main_dashboard_kpis`
Optimized: `hackathon`.`ecommerce_hypermarketing_dev_gold`.`category_demand_lift_current`
Optimized: `hackathon`.`ecommerce_hypermarketing_dev_gold`.`product_pushnow_current`
Optimized: `hackathon`.`ecomm

In [0]:

# COMMAND ----------

# ============================================================
# 9. Final validation display
# ============================================================

print("Setup complete.")
print("Gold schema:", GOLD)
print("Main KPI table:", qname(GOLD, "main_dashboard_kpis"))
print("Push Now table:", qname(GOLD, "product_pushnow_current"))
print("Action Center table:", qname(GOLD, "recommended_actions_current_hour"))

# COMMAND ----------

# MAGIC %md
# MAGIC ## Main KPIs

# COMMAND ----------

display(spark.table(qname(GOLD, "main_dashboard_kpis")).orderBy(F.desc("timestamp")).limit(1))

# COMMAND ----------

# MAGIC %md
# MAGIC ## Products to Push Now

# COMMAND ----------

display(
    spark.table(qname(GOLD, "product_pushnow_current"))
    .select("priority_rank", "sku", "product_name", "category", "demand_spike_pct", "cvr_pct", "roas", "stock_cover_hrs", "push_score", "action_type", "decision")
    .orderBy("priority_rank")
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Recommended Actions

# COMMAND ----------

display(spark.table(qname(GOLD, "recommended_actions_current_hour")).orderBy("priority_rank"))

# COMMAND ----------

# MAGIC %md
# MAGIC ## Data Quality Results

# COMMAND ----------

display(spark.table(qname(OPS, "data_quality_results")).orderBy(F.desc("checked_at")))


Setup complete.
Gold schema: hackathon.ecommerce_hypermarketing_dev_gold
Main KPI table: `hackathon`.`ecommerce_hypermarketing_dev_gold`.`main_dashboard_kpis`
Push Now table: `hackathon`.`ecommerce_hypermarketing_dev_gold`.`product_pushnow_current`
Action Center table: `hackathon`.`ecommerce_hypermarketing_dev_gold`.`recommended_actions_current_hour`


timestamp,hour_key,revenue_last_hour,orders_last_hour,units_last_hour,sessions_last_hour,conversion_rate_pct,roas,demand_pulse_index,stock_on_hand_total,created_at
2026-06-23T16:00:00Z,2026062316,38727.42,513,634,12378,4.144449830344159,5.050669163695359,156.0,2655,2026-06-23T16:38:07.483323Z


priority_rank,sku,product_name,category,demand_spike_pct,cvr_pct,roas,stock_cover_hrs,push_score,action_type,decision
1,EL-4821,Wireless Earbuds,Electronics,36.4,5.19,6.37,0.0,43.1,Guarded Push,PROMOTE_CAREFULLY_LOW_STOCK
2,BE-2044,Vitamin C Serum,Beauty,21.1,4.68,5.55,0.0,39.0,Guarded Push,PROMOTE_CAREFULLY_LOW_STOCK
3,FA-1108,Summer Linen Shirt,Fashion,29.4,4.27,5.12,0.0,37.2,Guarded Push,PROMOTE_CAREFULLY_LOW_STOCK
4,EL-5540,Smartwatch Lite,Electronics,27.4,4.57,5.43,0.0,36.6,Guarded Push,PROMOTE_CAREFULLY_LOW_STOCK
5,SP-1451,Yoga Mat Pro,Sports,17.4,3.3,4.02,0.0,29.7,Guarded Push,PROMOTE_CAREFULLY_LOW_STOCK
6,FA-2213,Running Shorts,Fashion,16.3,3.55,3.89,0.6,28.7,Guarded Push,PROMOTE_CAREFULLY_LOW_STOCK
7,HM-3175,Air Fryer 4L,Home,22.7,2.54,2.89,0.0,24.7,Guarded Push,PROMOTE_CAREFULLY_LOW_STOCK
8,GR-5632,Healthy Snack Box,Grocery,9.1,2.1,2.25,74.7,17.9,Secondary Push,SECONDARY_PUSH


priority_rank,sku,product_name,category,action_type,primary_channel_sequence,reason,suggested_budget_shift_pct,keep_promoting_until_hour,created_at
1,EL-4821,Wireless Earbuds,Electronics,Guarded Push,Retargeting Only → Frequency Cap → Await Restock,"Demand spike 36.4%, CVR 5.19%, ROAS 6.37x, stock cover 0.0 hrs",0.0,2026-06-23T19:00:00Z,2026-06-23T16:38:15.457794Z
2,BE-2044,Vitamin C Serum,Beauty,Guarded Push,Retargeting Only → Frequency Cap → Await Restock,"Demand spike 21.1%, CVR 4.68%, ROAS 5.55x, stock cover 0.0 hrs",0.0,2026-06-23T19:00:00Z,2026-06-23T16:38:15.457794Z
3,FA-1108,Summer Linen Shirt,Fashion,Guarded Push,Retargeting Only → Frequency Cap → Await Restock,"Demand spike 29.4%, CVR 4.27%, ROAS 5.12x, stock cover 0.0 hrs",0.0,2026-06-23T19:00:00Z,2026-06-23T16:38:15.457794Z
4,EL-5540,Smartwatch Lite,Electronics,Guarded Push,Retargeting Only → Frequency Cap → Await Restock,"Demand spike 27.4%, CVR 4.57%, ROAS 5.43x, stock cover 0.0 hrs",0.0,2026-06-23T19:00:00Z,2026-06-23T16:38:15.457794Z
5,SP-1451,Yoga Mat Pro,Sports,Guarded Push,Retargeting Only → Frequency Cap → Await Restock,"Demand spike 17.4%, CVR 3.3%, ROAS 4.02x, stock cover 0.0 hrs",0.0,2026-06-23T19:00:00Z,2026-06-23T16:38:15.457794Z


run_id,layer,table_name,rule_name,rule_status,failed_count,checked_at
1243a9ac-3bea-4c41-8b76-6dcc4052ae33,silver,fact_restock_schedule,"unique_key:sku,scheduled_restock_timestamp",PASS,0,2026-06-23T16:31:14.987252Z
1243a9ac-3bea-4c41-8b76-6dcc4052ae33,silver,fact_restock_schedule,not_null:scheduled_restock_timestamp,PASS,0,2026-06-23T16:31:13.820728Z
1243a9ac-3bea-4c41-8b76-6dcc4052ae33,silver,fact_restock_schedule,not_null:sku,PASS,0,2026-06-23T16:31:12.629883Z
1243a9ac-3bea-4c41-8b76-6dcc4052ae33,silver,fact_channel_performance_hourly,"unique_key:hour_key,channel",PASS,0,2026-06-23T16:31:07.007485Z
1243a9ac-3bea-4c41-8b76-6dcc4052ae33,silver,fact_channel_performance_hourly,not_null:channel,PASS,0,2026-06-23T16:31:05.865226Z
1243a9ac-3bea-4c41-8b76-6dcc4052ae33,silver,fact_channel_performance_hourly,not_null:hour_key,PASS,0,2026-06-23T16:31:04.657194Z
1243a9ac-3bea-4c41-8b76-6dcc4052ae33,silver,fact_category_performance_hourly,"unique_key:hour_key,category",PASS,0,2026-06-23T16:30:57.796041Z
1243a9ac-3bea-4c41-8b76-6dcc4052ae33,silver,fact_category_performance_hourly,not_null:category,PASS,0,2026-06-23T16:30:56.807489Z
1243a9ac-3bea-4c41-8b76-6dcc4052ae33,silver,fact_category_performance_hourly,not_null:hour_key,PASS,0,2026-06-23T16:30:55.660309Z
1243a9ac-3bea-4c41-8b76-6dcc4052ae33,silver,fact_orders_hourly_summary,unique_key:hour_key,PASS,0,2026-06-23T16:30:49.041296Z
